## Import packages

In [ ]:
import sys
print(sys.version)  # Print Python version in use

import numpy as np                                      # Array operations
import matplotlib.pyplot as plt                         # Plotting library
import torch                                            # Main PyTorch library
import torch.optim as optim                             # Optimization algorithms
import torch.nn as nn                                   # Neural network modules
import copy                                             # For copying PyTorch objects
import os                                               # Operating system utilities
import glob                                             # Pattern matching
import pandas as pd                                     # Data manipulation
import imageio.v2 as imageio                            # Image manipulation - for creating GIFs
import rasterio                                         # Raster data handling
import folium                                           # Interactive maps
import re                                               # Regular expressions

from torch.utils.data import Dataset, DataLoader        # Dataset and batch data loading
from rasterio.plot import show                           # Plot raster data
from IPython.display import Image, display              # For plotting GIFs
from datetime import datetime, timedelta                # Date/time utilities
from tqdm import tqdm                                   # Progress bar
from pyproj import Transformer                          # Coordinate transformation

import deepSSF_model_scalar_move as deepSSF_model # Import the .py file containing the deepSSF model     
import deepSSF_dataset_crop_training_functions as deepSSF_training_functions # Import the .py file containing the training functions
import deepSSF_loss_pxpy_mixedLR as deepSSF_loss        # Import the .py file containing the deepSSF loss function
import deepSSF_early_stopping                           # Import the .py file containing the early stopping function  
import deepSSF_utils                                    # Import the .py file containing the utility functions 

base_path = '..'
print(f"Using base path: {base_path}")

## Set the device (accelerator - cuda for NVIDIA GPU or mps for Mac)

In [ ]:
# Set the device to be used (GPU or CPU)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():  # For Mac M1/M2/M3/M4
    device = "mps"
else:
    device = "cpu"
    
print(f"Using {device} device")

if torch.backends.mps.is_available():
    # Set default tensor type for PyTorch
    torch.set_default_dtype(torch.float32)
    print('Set default tensor type to float32')

## Set up the script

In [ ]:
# Get today's date
today_date = datetime.today().strftime('%Y-%m-%d')

# Set random seed for reproducibility
seed = 42

# Size of the input window - must be odd so there is a central pixel
window_size = 151  # in cells/pixels
pixel_size = 25  # in meters

# Which dataset?
dataset_area = 'mimal_gropyul'

### Create a directory to save the outputs

If we have already run this code today, we will add update index to create a new folder

In [ ]:
# Count existing directories with similar pattern
pattern = f'{base_path}/Python/outputs/model_training/{dataset_area}_S2_nxn{window_size}_polar_move_*_{today_date}'
existing_dirs = glob.glob(pattern)
dir_index = len(existing_dirs) + 1

print(f"Existing directories matching pattern '{pattern}': {existing_dirs}")
print(f"Next directory index: {dir_index}")

# Create directory with index
output_dir = f'{base_path}/Python/outputs/model_training/{dataset_area}_S2_nxn{window_size}_polar_move_{dir_index}_{today_date}'
os.makedirs(output_dir, exist_ok=True)

print(f"Created directory: {output_dir}")


### To use an existing directory for loading trained model

In [ ]:
# output_dir = f'{base_path}/Python/outputs/model_training_S2/djelk_S2_nxn151_CNN_move_1_2025-06-15'
# print(f"Output directory: {output_dir}")

# Load the GPS data for training

In [ ]:
# Load data into dataset
csv_file = f'{base_path}/data/{dataset_area}_steps.csv'
print(f"Loading data from: {csv_file}")

training_df = pd.read_csv(csv_file)
print(f"Data loaded with {len(training_df)} rows and {len(training_df.columns)} columns")

# For testing, use a subset of the data
# training_df = training_df[:1000] # first n rows
# training_df = training_df[training_df['id'] == 2005] # first ID

# # Subset by the time period
# start_date = '2023-01-01'
# end_date = '2023-12-31'
# training_df['date'] = pd.to_datetime(training_df['t1_'])
# training_df = training_df[(training_df['date'] >= start_date) & (training_df['date'] <= end_date)]

# Remove any steps that are outside the window size
training_df = training_df[training_df['dx'] < (window_size-1)*pixel_size/2]  # Assuming dx is in meters, and window_size is in cells/pixels
training_df = training_df[training_df['dx'] > -(window_size-1)*pixel_size/2]  # Assuming dx is in meters, and window_size is in cells/pixels
training_df = training_df[training_df['dy'] < (window_size-1)*pixel_size/2]  # Assuming dy is in meters, and window_size is in cells/pixels
training_df = training_df[training_df['dy'] > -(window_size-1)*pixel_size/2]  # Assuming dy is in meters, and window_size is in cells/pixels

# Number of rows after subsetting
print(f"Number of rows after subsetting by window size: {len(training_df)}")

training_df.head()

## Check the distribution of fix-intervals

We see that there are steps with massive time intervals between fixes (1000's of hours), likely due to tags running out of battery but then recharging at a later date.

In [ ]:
plt.hist(training_df['dt_hour'], bins=100)
plt.xlabel('Time difference (hours)')
plt.ylabel('Frequency')
plt.title('Distribution of Time Difference (Hours)')

How many steps have an interval greater than 48 hours?

In [ ]:
percentage_hours = len(training_df[training_df['dt_hour'] > 48]) / len(training_df) * 100
print(f"{percentage_hours:.3f}%")  # Percentage of hours > 48

# Filter out rows with dt_hour > 48
training_df = training_df[training_df['dt_hour'] <= 48]

There are only about 2.6% of steps that have an interval greater than 48 hours, so we will remove them. This will essentially split the trajectory up into bursts, with sections that have fix-intervals less than 48 hours.

In [ ]:
plt.hist(training_df['dt_hour'], bins=100)
plt.xlabel('Time Difference (hours)')
plt.ylabel('Frequency')
plt.title('Distribution of Time Difference (Hours)')

In [ ]:
plt.hist(training_df['hour_t1'], bins=100)
plt.xlabel('Hour of the day')
plt.ylabel('Frequency')
plt.title('Distribution of Hour of the Day')

### Distribution of step lengths

In [ ]:
plt.hist(training_df['sl_'], bins=100)
plt.xlabel('Step length (m)')
plt.ylabel('Frequency')
plt.title('Distribution of Step Lengths (m)')
plt.show()

plt.hist(np.log(training_df['sl_']), bins=100)
plt.xlabel('Log of Step length (m)')
plt.ylabel('Frequency')
plt.title('Distribution of Log of Step Lengths (m)')
plt.show()

### Distribution of turn angles

In [ ]:
plt.hist(training_df['ta_'], bins=100)
plt.xlabel('Turn angles (radians)')
plt.ylabel('Frequency')
plt.title('Distribution of Turn Angle (radians)')

In [ ]:
training_df

In [ ]:
timestamp = training_df['t1_'].iloc[0]  # Get the first timestamp
timestamp = datetime.fromisoformat(timestamp.replace('Z', '+00:00'))

year = timestamp.year
month = timestamp.month

print(f"Year: {year}")    # Output: Year: 2023
print(f"Month: {month}")  # Output: Month: 8

# Set up functions for training

To get the validation running we need a few extra functions.

Firstly, we need to index the Sentinel-2 layers correctly, based on the time of the simulated location. We'll do this by creating a function that takes day of the year of the simulated location and returns the correct index for the Sentinel-2 layers.

This indexing is slightly different from the indexing we used for the `deepSSF_simulations.ipynb` notebook, which was indexing NDVI layers. In that case we were indexing the layers directly, and therefore the first entry was at 0 (i.e., March was in month_index = 2). Here, we are creating a string that corresponds to the layer name, and therefore the first entry is at 1. (i.e., March will be at month_index = 3)

In [ ]:
def extract_year_month_regex(filename):
    pattern = r'(\d{4}_\d{2})'
    match = re.search(pattern, filename)
    return match.group(1) if match else None

def extract_datetime_regex(filename):
    pattern = r'(\d{4}_\d{2})'
    match = re.search(pattern, filename)
    if match:
        year_month_str = match.group(1)
        # Convert "YYYY_MM" to datetime object
        year, month = year_month_str.split('_')
        return datetime(int(year), int(month), 15)
    return "Datetime string not found in filename"

def load_s2_data(s2_dir):
    """
    Load S2 TIFF files from directory.

    Args:
        s2_dir: Directory containing S2 TIFF files
        
    Returns:
        dict: Dictionary with date strings as keys and numpy arrays as values
    """
    print(f"Loading S2 data from {s2_dir}")

    raster_transform = None  # Ensure raster_transform is always defined
    
    # Use glob to get a list of all TIFF files matching the pattern
    tif_files = glob.glob(os.path.join(s2_dir, f'S2_*.tif'))
    print(f'Found {len(tif_files)} S2 TIFF files')
    
    # Initialize dictionary to store data with date as key
    s2_data_dict = {}
    
    # Loop over each TIFF file to read and process the data
    for tif_file in tif_files:
        filename = os.path.basename(tif_file)
        date_str = extract_year_month_regex(filename)
        print(f"Loading {filename} -> {date_str}")
        
        with rasterio.open(tif_file) as src:
            data = src.read()
            n_nan = np.isnan(data).sum()
            print(f"  NaN values: {n_nan} ({n_nan / data.size:.4%})")
            data = np.nan_to_num(data, nan=0) # Replace NaNs with 0
            data = data / 10000.0 # Scale by 10000
            s2_data_dict[date_str] = data
            raster_transform = src.transform
    
    print(f"Loaded {len(s2_data_dict)} S2 datasets")

    if raster_transform is None:
        raise ValueError("raster_transform is not set. Please check your input layer paths and ensure at least one TIFF file is provided.")
    
    return s2_data_dict, raster_transform

def day_to_month_index(day_of_year):
    """
    Convert day of year to month index for S2 data selection.
    
    Args:
        day_of_year: Day of the year (1-365)
        
    Returns:
        int: Month index (1-12)
    """
    base_date = datetime(2019, 1, 1)
    date = base_date + timedelta(days=int(day_of_year % 365) - 1)
    year_diff = date.year - base_date.year
    month_index = (date.month) + (year_diff * 12)
    if month_index == 0:
        month_index += 1
    return month_index

def subset_layer_vectorized(layer_data, px, py, window_size):
    """
    Efficiently subset a layer (works for both 2D and 3D arrays).
    
    Args:
        layer_data: numpy array, either 2D [H, W] or 3D [bands, H, W]
        px, py: pixel coordinates
        window_size: Size of the cropping window
        
    Returns:
        torch.Tensor: cropped patch
    """
    half_window = window_size // 2
    
    # Calculate window boundaries
    row_start = py - half_window
    row_stop = py + half_window + 1
    col_start = px - half_window
    col_stop = px + half_window + 1
    
    # Handle 2D vs 3D arrays
    if layer_data.ndim == 2:
        height, width = layer_data.shape
        subset = np.full((window_size, window_size), -1.0, dtype=layer_data.dtype)
        
        # Calculate valid regions
        valid_row_start = max(0, row_start)
        valid_row_stop = min(height, row_stop)
        valid_col_start = max(0, col_start)
        valid_col_stop = min(width, col_stop)
        
        # Calculate subset indices
        subset_row_start = valid_row_start - row_start
        subset_row_stop = subset_row_start + (valid_row_stop - valid_row_start)
        subset_col_start = valid_col_start - col_start
        subset_col_stop = subset_col_start + (valid_col_stop - valid_col_start)
        
        # Copy valid region
        if valid_row_start < valid_row_stop and valid_col_start < valid_col_stop:
            subset[subset_row_start:subset_row_stop, subset_col_start:subset_col_stop] = \
                layer_data[valid_row_start:valid_row_stop, valid_col_start:valid_col_stop]
                
    else:  # 3D array [bands, height, width]
        n_bands, height, width = layer_data.shape
        subset = np.full((n_bands, window_size, window_size), -1.0, dtype=layer_data.dtype)
        
        # Calculate valid regions
        valid_row_start = max(0, row_start)
        valid_row_stop = min(height, row_stop)
        valid_col_start = max(0, col_start)
        valid_col_stop = min(width, col_stop)
        
        # Calculate subset indices
        subset_row_start = valid_row_start - row_start
        subset_row_stop = subset_row_start + (valid_row_stop - valid_row_start)
        subset_col_start = valid_col_start - col_start
        subset_col_stop = subset_col_start + (valid_col_stop - valid_col_start)
        
        # Vectorized copy for all bands
        if valid_row_start < valid_row_stop and valid_col_start < valid_col_stop:
            subset[:, subset_row_start:subset_row_stop, subset_col_start:subset_col_stop] = \
                layer_data[:, valid_row_start:valid_row_stop, valid_col_start:valid_col_stop]
    
    # Returns the subset as a PyTorch tensor, and the start indices for rows and columns
    return torch.from_numpy(subset.copy()).float(), col_start, row_start

def load_environmental_layers(layer_paths):
    """
    Load all environmental layers from the provided paths.
    
    Args:
        layer_paths: Dict containing paths to environmental layers
        
    Returns:
        dict: Dictionary with layer names as keys and numpy arrays as values
    """
    environmental_layers = {}
    raster_transform = None  # Ensure raster_transform is always defined
    
    # Load S2 data from TIFF files
    if 's2_dir' in layer_paths:
        environmental_layers['s2'], raster_transform = load_s2_data(layer_paths['s2_dir'])
    
    # Load other layers
    for layer_name, layer_path in layer_paths.items():
        if layer_name not in ['s2', 's2_dir']:
            if isinstance(layer_path, str) and layer_path.endswith('.tif'):
                # Load TIFF file
                with rasterio.open(layer_path) as src:
                    data = src.read()
                    data = np.nan_to_num(data, nan=0)

                    # Scale data to [0, 1] 
                    max_value = np.nanmax(data)
                    min_value = np.nanmin(data)
                    print(f"Layer: {layer_name}, Min: {min_value}, Max: {max_value}")
                    print("Layer scaled to [0, 1]")
                    data = (data - min_value) / max_value
                    
                    if data.shape[0] == 1:
                        data = data[0]
                    environmental_layers[layer_name] = data
                    # Only set raster_transform if it hasn't been set yet
                    if raster_transform is None:
                        raster_transform = src.transform
            else:
                # Load numpy file
                environmental_layers[layer_name] = np.load(layer_path, mmap_mode='r')
    
    if raster_transform is None:
        raise ValueError("raster_transform is not set. Please check your input layer paths and ensure at least one TIFF file is provided.")
    
    return environmental_layers, raster_transform

def precompute_coordinates_and_months(movement_df, raster_transform):
    """
    Pre-compute pixel coordinates and S2 month selection for efficiency.
    
    Args:
        movement_df: DataFrame with GPS locations
        raster_transform: Raster transform for coordinate conversion
        
    Returns:
        tuple: (pixel_coords, s2_months) lists
    """
    pixel_coords = []
    s2_months = []
    
    for idx, row in movement_df.iterrows():
        # Pixel coordinates
        x, y = row['x1_'], row['y1_']
        px, py = ~raster_transform * (x, y)
        px, py = int(np.floor(px)), int(np.floor(py))
        pixel_coords.append((px, py))
        
        # # S2 month selection based on day of year
        # yday = row['yday_t1']
        # month_index = day_to_month_index(yday)
        # selected_month = f'2019_{month_index:02d}'
        # s2_months.append(selected_month)

        # S2 month selection based on t1 timestamp
        timestamp = row['t1_']
        dt = datetime.fromisoformat(timestamp.replace('Z', '+00:00'))
        year = dt.year
        month = dt.month
        print(f"Year: {year}")    # Output: Year: 2023
        print(f"Month: {month}")  # Output: Month: 8
        selected_month = f'{year}_{month:02d}'
        s2_months.append(selected_month)
    
    return pixel_coords, s2_months

def landscape_crop(movement_df, environmental_layers, raster_transform, window_size):
    """
    Create a dataset of landscape crops for all buffalo observations.
    
    Args:
        movement_df: DataFrame with movement observations
        environmental_layers: Dictionary with environmental layers
        raster_transform: Raster transform for coordinate conversion
        window_size: Size of the cropping window
        
    Returns:
        Dataset: Custom dataset with landscape crops
    """
    # pixel_coords, _ = precompute_coordinates_and_months(movement_df, raster_transform)

    x1y1_pixel_coords = []

    for idx, row in movement_df.iterrows():
        # Pixel coordinates for each location
        x1, y1 = row['x1_'], row['y1_']
        px1, py1 = ~raster_transform * (x1, y1)
        px1, py1 = int(np.floor(px1)), int(np.floor(py1))
        x1y1_pixel_coords.append((px1, py1))
    
    # Calculate the extent from the min and max pixel coordinates
    min_px = min(px for px, _ in x1y1_pixel_coords)
    max_px = max(px for px, _ in x1y1_pixel_coords)
    min_py = min(py for _, py in x1y1_pixel_coords)
    max_py = max(py for _, py in x1y1_pixel_coords)

    # Crop the environmental layers to the extent of the buffalo observations
    # with a buffer of half the window size
    buffer = window_size // 2

    cropped_layers = {}
    
    for layer_name, layer_data in environmental_layers.items():
        if layer_data.ndim == 2:  # 2D layer
            cropped_layer = layer_data[min_py-buffer:max_py+buffer+1, min_px-buffer:max_px+buffer+1]
        else:  # 3D layer
            cropped_layer = layer_data[:, min_py-buffer:max_py+buffer+1, min_px-buffer:max_px+buffer+1]
        
        cropped_layers[layer_name] = torch.from_numpy(cropped_layer.copy())

    print(f"Cropped layers to extent: ({min_px-buffer}, {min_py-buffer}) to ({max_px+buffer}, {max_py+buffer})")
    # print(f"Cropped layers shapes: {[layer.shape for layer in cropped_layers.values()]}")

    return cropped_layers


# Dataloader

In [ ]:
class MovementDataset(Dataset):
    def __init__(self, movement_df, layer_paths, window_size):
        """
        Simplified dataset class with external utility functions.
        
        Args:
            movement_df: DataFrame with GPS locations and scalar features
            layer_paths: Dict containing paths to environmental layers
            window_size: Size of the cropping window
            raster_transform: Raster transform for coordinate conversion
        """
        self.movement_df = movement_df.reset_index(drop=True) # Ensure clean indexing
        self.window_size = window_size
        
        # Load environmental layers using external function
        self.environmental_layers, self.raster_transform = load_environmental_layers(layer_paths)

        # # Crop the environmental layers to the extent of buffalo observations
        # self.environmental_layers = landscape_crop(
        #     self.movement_df, 
        #     self.environmental_layers, 
        #     self.raster_transform, 
        #     self.window_size
        # )
        
        # Pre-convert scalar data to tensors
        self.scalar_to_grid_data = torch.from_numpy(
            movement_df[['hour_t1_sin1', 
                        'hour_t1_cos1', 
                        'yday_t1_sin1', 
                        'yday_t1_cos1',
                        'dt_hour']].values).float() # called from the csv dataframe
        

        self.bearing_tm1 = pd.to_numeric(movement_df['bearing'], errors='coerce').shift(1).fillna(0)
        self.bearing_tm1 = torch.from_numpy(self.bearing_tm1.values).unsqueeze(1).float()

        self.x1y1_pixel_coords = []
        self.x2y2_pixel_coords = []
        self.s2_months = []

        # iterrows() returns the index and row values, 
        # but we just need the values 
        for idx, row in self.movement_df.iterrows():
            
            # Pixel coordinates for each location
            x1, y1 = row['x1_'], row['y1_']
            px1, py1 = ~self.raster_transform * (x1, y1)
            px1, py1 = int(np.floor(px1)), int(np.floor(py1))
            self.x1y1_pixel_coords.append((px1, py1))

            # Pixel coordinates for next location
            x2, y2 = row['x2_'], row['y2_']
            px2, py2 = ~self.raster_transform * (x2, y2)
            px2, py2 = int(np.floor(px2)), int(np.floor(py2))
            self.x2y2_pixel_coords.append((px2, py2))

            # # S2 month selection based on day of year
            # yday = row['yday_t1']
            # month_index = day_to_month_index(yday)
            # selected_month = f'2019_{month_index:02d}'
            # self.s2_months.append(selected_month)

            # # S2 month selection based on t1 timestamp - requires hard-coding dataset_area=='djelk'
            # timestamp = row['t1_']
            # dt = datetime.fromisoformat(timestamp.replace('Z', '+00:00'))
            # if dataset_area == 'djelk':
            #     # For Djelk, we only have 2019 layers, 
            #     # so we will use them for data from 2018 or 2019
            #     year = '2019'
            #     month = dt.month
            # else:
            #     year = dt.year
            #     month = dt.month
            # selected_month = f'{year}_{month:02d}'
            # self.s2_months.append(selected_month)

            # S2 month selection based on t1 timestamp - if exact year and month not available, 
            # select the same month but in the previous year, if that is not available, select the next year
            timestamp = row['t1_']
            dt = datetime.fromisoformat(timestamp.replace('Z', '+00:00'))
            year = dt.year
            month = dt.month
            selected_month = f'{year}_{month:02d}'
            self.s2_months.append(selected_month)

        self.n_samples = len(movement_df)


    def __len__(self):
        return self.n_samples

    def __getitem__(self, index):

        # Get precomputed pixel coordinates and S2 month
        px1, py1 = self.x1y1_pixel_coords[index]
        px2, py2 = self.x2y2_pixel_coords[index]
        selected_s2_month = self.s2_months[index]
        
        # Crop environmental layers
        cropped_layers = []
        
        # Handle S2 data with temporal selection
        if 's2' in self.environmental_layers:
            if selected_s2_month in self.environmental_layers['s2']:
                s2_data = self.environmental_layers['s2'][selected_s2_month]
            else:
                # Fallback to previous year
                year = int(selected_s2_month.split('_')[0]) - 1
                month = selected_s2_month.split('_')[1]
                fallback_month = f'{year}_{month}'
                if fallback_month in self.environmental_layers['s2']:
                    print(f"Month {selected_s2_month} not available, using {fallback_month}")
                    s2_data = self.environmental_layers['s2'][fallback_month]
                else:
                    # Fallback to next year
                    year += 2
                    fallback_month = f'{year}_{month}'
                    if fallback_month in self.environmental_layers['s2']:
                        print(f"Month {selected_s2_month} not available, using {fallback_month}")
                        s2_data = self.environmental_layers['s2'][fallback_month]
                    else:
                        # If still not found, use the first available month
                        fallback_month = list(self.environmental_layers['s2'].keys())[0]
                        print(f"Month {selected_s2_month} not available, using {fallback_month}")
                        s2_data = self.environmental_layers['s2'][fallback_month]

            s2_crop, col_start, row_start = subset_layer_vectorized(s2_data, px1, py1, self.window_size)
            cropped_layers.append(s2_crop)
        
        # Handle other layers
        for layer_name, layer_data in self.environmental_layers.items():
            if layer_name not in ['s2']:
                layer_crop, col_start, row_start = subset_layer_vectorized(layer_data, px1, py1, self.window_size)
                if layer_crop.ndim == 2:
                    layer_crop = layer_crop.unsqueeze(0)
                cropped_layers.append(layer_crop)
        
        # Combine all environmental layers
        if len(cropped_layers) > 1:
            spatial_data = torch.cat(cropped_layers, dim=0)
        else:
            spatial_data = cropped_layers[0]

        # Subtract the col_start and row_start from the pixel coordinates
        x2y2_pixel_coords = (px2 - col_start, py2 - row_start)
        # x1y1_pixel_coords = (px1 - col_start, py1 - row_start)

        return (spatial_data, 
                self.scalar_to_grid_data[index], 
                self.bearing_tm1[index], 
                x2y2_pixel_coords,
                self.raster_transform
                # x1y1_pixel_coords
                )

In [ ]:
layer_paths = {
    's2_dir': f'{base_path}/mapping/cropped rasters/{dataset_area}/',  # Directory containing S2 TIFFs
    'elevation': f'{base_path}/mapping/cropped rasters/{dataset_area}/DEM_{dataset_area}_reprojected.tif',  # elevation .tif file
    'slope': f'{base_path}/mapping/cropped rasters/{dataset_area}/slope_{dataset_area}.tif',  # slope .tif file
}

In [ ]:
pretraining_dataset = MovementDataset(
    movement_df=training_df,
    layer_paths=layer_paths,
    window_size=window_size
)

In [ ]:
# single_layer = pretraining_dataset.environmental_layers['s2']['2024_01'][0, :, :]  # Example layer from S2 data
single_layer = pretraining_dataset.environmental_layers['elevation']  # other enviro layer
print(f"Single layer shape: {single_layer.shape}")

plt.imshow(single_layer, cmap='viridis')
plt.colorbar()
plt.close()

In [ ]:
training_split = 0.8
validation_split = 0.1
test_split = 0.1

pretraining_dataset_train, pretraining_dataset_val, pretraining_dataset_test = torch.utils.data.random_split(pretraining_dataset, [training_split, validation_split, test_split])
print(len(pretraining_dataset_train))
print(len(pretraining_dataset_val))
print(len(pretraining_dataset_test))

### Create dataloaders

In [ ]:
batch_size = 32 # batch size
num_workers = 0 # 0 is recommended for Jupyter notebooks
# num_workers = min(os.cpu_count() - 1, 8) # number of workers for data loader
print("Number of CPU cores available:", os.cpu_count())
print("Number of workers for data loader:", num_workers)

pretraining_dataloader_train = DataLoader(dataset=pretraining_dataset_train, 
                              batch_size=batch_size, 
                              shuffle=True, 
                              num_workers=num_workers
                            #   pin_memory=torch.accelerator.is_available(),  # Only if using GPU
                              # persistent_workers=True  # Keeps workers alive between epochs
)
pretraining_dataloader_val = DataLoader(dataset=pretraining_dataset_val, 
                             batch_size=batch_size, 
                             shuffle=True, 
                             num_workers=num_workers
                            #  pin_memory=torch.accelerator.is_available(),  # Only if using GPU
                             # persistent_workers=True  # Keeps workers alive between epochs
)
pretraining_dataloader_test = DataLoader(dataset=pretraining_dataset_test, 
                              batch_size=batch_size, 
                              shuffle=True, 
                              num_workers=num_workers
                            #   pin_memory=torch.accelerator.is_available(),  # Only if using GPU
                              # persistent_workers=True  # Keeps workers alive between epochs
)

## Check that the dataset and dataloader are working

In [ ]:
# Display image and label.
spatial_data, scalar_to_grid_data, bearing_tm1, labels, raster_transform = next(iter(pretraining_dataloader_train))
print(f"Feature x1 batch shape:             {spatial_data.size()}")
print(f"Feature x2 batch shape:             {scalar_to_grid_data.size()}")
print(f"Feature x3 batch shape:             {bearing_tm1.size()}")
print(f'First bearing_tm1:                  {bearing_tm1[0]}')
x2, y2 = labels
print(f"Labels batch shape:                 {labels[0].size()}")
print(f'Resolution of the raster:           {raster_transform[0][0]} m/pixel')

# print(x3.detach().numpy())
print(f'Scalar covariates:                  {scalar_to_grid_data[0,:]}')

# Convert the PyTorch tensor to a NumPy array for plotting
# Detach from the computation graph, move to CPU, and convert to NumPy
spatial_data_np = spatial_data.detach().cpu().numpy()

# # If checking the pixel coordinates for the current location
# spatial_data, scalar_to_grid_data, bearing_tm1, labels, x1y1 = next(iter(pretraining_dataloader_train))
# x1, y1 = x1y1
# # Print the pixel coordinates for the current location
# print(f"Pixel coordinates for current location: {x1[0]}, {y1[0]}")
# spatial_data_np[0, :, y1[0], x1[0]] = -np.inf  # Mask out the current location

# Print the pixel coordinates for the next step
print(f"Pixel coordinates for next step:    {x2[0]}, {y2[0]}")

# Plot the subset
fig, axs = plt.subplots(2, 2, figsize=(10, 10))

# mask out the current location to check that it's in the middle
# mask out the next step pixel - y-coordinate (rows) before x-coordinate (columns)
spatial_data_np[0, :, y2[0], x2[0]] = -np.inf

# Set the colourmap to 'viridis' for better visibility
# And set NaN values to red for visibility
# Create a copy of the colormap and set the color for bad/NaN values
cmap = plt.get_cmap('viridis').copy()
cmap.set_bad(color='red')  # Set NaN values to red (or any color you want)

# Plot each band in a separate subplot
im0 = axs[0, 0].imshow(spatial_data_np[0,0,:,:], cmap=cmap)
axs[0, 0].set_title('Local S2 Band 1')
fig.colorbar(im0, ax=axs[0, 0], shrink = 0.75)
im1 =axs[0, 1].imshow(spatial_data_np[0,1,:,:], cmap=cmap)
axs[0, 1].set_title('Local S2 Band 2')
fig.colorbar(im1, ax=axs[0, 1], shrink = 0.75)
im2 = axs[1, 0].imshow(spatial_data_np[0,12,:,:], cmap=cmap)
axs[1, 0].set_title('Local Elevation')
fig.colorbar(im2, ax=axs[1, 0], shrink = 0.75)
im3 = axs[1, 1].imshow(spatial_data_np[0,13,:,:], cmap=cmap)
axs[1, 1].set_title('Local Slope')
fig.colorbar(im3, ax=axs[1, 1], shrink = 0.75)

# # Also plot the target as a single plot
# fig, ax = plt.subplots(figsize=(5, 5))
# im = ax.imshow(labels.detach().numpy()[0,:,:], cmap='viridis')
# ax.set_title('Target location (next step)')

# Load the model

As we have already described the model in detail in the `deepSSF_model` script, we can simply import the model here.

We will use the same model architecture as in the previous script, except that we will need to use a slightly edited dictionary to account for the additional input channels.

## Define the parameters for the model

Here we enter the specific parameter values and hyperparameters for the model. 
These are the values that will be used to instantiate the model.

In [ ]:
# Get the number of covariates in the spatial and scalar data
spatial_data, scalar_to_grid_data, bearing_tm1, labels, raster_transform = next(iter(pretraining_dataloader_train))
n_spatial_layers = spatial_data.size(1)  # Number of spatial layers (bands)
n_scalar_inputs = scalar_to_grid_data.size(1)  # Number of scalar inputs
print(f"Number of spatial layers: {n_spatial_layers}") # Print the number of spatial layers
print(f"Number of scalar inputs: {n_scalar_inputs}") # Print the number of scalar inputs

# Model specific information
# These values are used to determine the number of inputs entering the fully connected block
# THIS IS NOT WHERE THESE PARAMETERS ARE SET, THEY ARE SET IN THE MODEL DEFINITION, which is in the deepSSF_model_scalar_move.py file
n_max_pool_layers = 2 # used to determine the number of inputs entering the fully connected block - needs to be manually changed if the number of max pooling layers is changed
n_conv_layers = 4 # number of convolution layers in the CNN model

params_dict = {"batch_size": batch_size, #number of samples in each batch
               "image_dim": window_size, #number of pixels along the edge of each local patch/image
               "pixel_size": 25, #number of metres along the edge of a pixel
               "input_channels": n_spatial_layers + n_scalar_inputs, #number of spatial layers in each image + number of scalar layers that are converted to a grid
               "dim_in_nonspatial_to_grid": n_scalar_inputs, #the number of scalar predictors that are converted to a grid and appended to the spatial features
               "dense_dim_in_nonspatial": n_scalar_inputs, #change this to however many other scalar predictors you have (bearing, velocity etc)
               "kernel_size": 3, #the size of the 2D moving windows / kernels that are being learned
               "stride": 1, #the stride used when applying the kernel.  This reduces the dimension of the output if set to greater than 1
               "kernel_size_mp": 2, #the size of the kernel that is used in max pooling operations
               "stride_mp": 2, #the stride that is used in max pooling operations
               "padding": 1, #the amount of padding to apply to images prior to applying the 2D convolution
               "num_movement_params": 12, #number of parameters used to parameterise the movement kernel
               "dropout": 0.1, #the proportion of nodes that are dropped out in the dropout layers

               # hyperparameters that change the model architecture
               "output_channels": 8, #number of convolution filters to learn
               "output_channels_movement": 4, #number of convolution filters to learn for the movement kernel
               "dense_dim_hidden": 128, #number of nodes in the hidden layers

               # this will be updated below
               "dense_dim_in_all": n_scalar_inputs, #number of inputs entering the fully connected block once the nonspatial features have been concatenated to the spatial features
               "device": device
               }

# Now update the dictionary with calculated values
# params_dict["dense_dim_in_all"] = int(((params_dict["image_dim"] - (params_dict["image_dim"] % 2))**2) * (params_dict["output_channels_movement"] / (4**n_max_pool_layers)))

## Instantiate the model

As described in the `deepSSF_train.ipynb` script, we saved the model definition into a file named `deepSSF_model.py`. We can instantiate the model by importing the file (which was done when importing other packages) and calling the classes parameter dictionary from that script.

In [ ]:
params = deepSSF_model.ModelParams(params_dict)
model_pretraining = deepSSF_model.ConvJointModel(params).to(device)
print(model_pretraining)

# Pull out some testing data

To test the other blocks, and the full model, we will need some data. We can pull that out from the training set.

In [ ]:
# Number of samples in the train dataset
print("Number of samples in the train dataset: ", len(pretraining_dataset_train))
print('\n')

# Select an index from the test dataset to retrieve a sample, between 0 and number of samples
# We picked this fairly arbitrarily, but with some interesting environmental features to illustrate the model's predictions
iteration_index = 100

# iteration_index = 1010
# iteration_index = 3971

# print(movement_df.iloc[iteration_index])

# return spatial_data_x, scalar_to_grid_data, bearing_tm1, target

# 2. Retrieve a single sample (features and label) from the test dataset at the specified index

# sample_spatial_covs is a sample of the spatial covariates for a single step
# sample_temporal_covs is a sample of the temporal covariates for a single step
# sample_prev_bearing is a sample bearing of the previous step
# sample_next_step is the target label (what we are trying to predict) for the next step

# We set these here and will also use them later in the script to check how the model's predictions look,
# and when we extract feature maps from the convolutional layers
sample_spatial_covs, sample_temporal_covs, sample_prev_bearing, sample_next_step, raster_transform = pretraining_dataloader_train.dataset[iteration_index]

# 3. Reshape data tensors to add a batch dimension (since the model expects batches)
sample_spatial_covs = sample_spatial_covs.unsqueeze(0).to(device)
sample_temporal_covs = sample_temporal_covs.unsqueeze(0).to(device)
sample_prev_bearing = sample_prev_bearing.unsqueeze(0).to(device)

print(f'Shape of the sample spatial covariates:  {sample_spatial_covs.shape}')
print(f'Shape of the sample temporal covariates: {sample_temporal_covs.shape}')
print(f'Shape of the sample previous bearing:    {sample_prev_bearing.shape}')
print(f'Location of sample next step (px, py):   {sample_next_step}')

In [ ]:
# Print the pixel coordinates for the next step
print(f'Location of sample next step (px, py):   {sample_next_step}')

# Plot the subset
fig, axs = plt.subplots(2, 2, figsize=(10, 10))

sample_spatial_covs_np = sample_spatial_covs.detach().cpu().numpy()
sample_spatial_covs_np[0, :, sample_next_step[1], sample_next_step[0]] = -np.inf
# Set the colourmap to 'viridis'
cmap = plt.get_cmap('viridis')
cmap.set_bad(color='red')  # Set NaN values to red (or any color you want)

axs[0, 0].imshow(sample_spatial_covs_np[0,0,:,:], cmap=cmap)
axs[0, 0].set_title('Local S2 Band 1')
axs[0, 1].imshow(sample_spatial_covs_np[0,1,:,:], cmap=cmap)
axs[0, 1].set_title('Local S2 Band 2')
axs[1, 0].imshow(sample_spatial_covs_np[0,12,:,:], cmap=cmap)
axs[1, 0].set_title('Local Elevation')
axs[1, 1].imshow(sample_spatial_covs_np[0,13,:,:], cmap=cmap)
axs[1, 1].set_title('Local Slope')

### Pull out the scalar values

In [ ]:
# Extract the first sample (index 0) and its respective channel for each variable:
hour_t1_sin = sample_temporal_covs.detach().cpu().numpy()[0, 0]
hour_t1_cos = sample_temporal_covs.detach().cpu().numpy()[0, 1]
yday_t1_sin = sample_temporal_covs.detach().cpu().numpy()[0, 2]
yday_t1_cos = sample_temporal_covs.detach().cpu().numpy()[0, 3]

# Convert x3 similarly and extract the bearing from the first sample and channel:
bearing = sample_prev_bearing.detach().cpu().numpy()[0, 0]

hour_t1 = deepSSF_utils.recover_hour(hour_t1_sin, hour_t1_cos)
hour_t1_integer = int(hour_t1)  # Convert to integer
print(f'Hour:               {hour_t1_integer}')

yday_t1 = deepSSF_utils.recover_yday(yday_t1_sin, yday_t1_cos)
yday_t1_integer = int(yday_t1)  # Convert to integer
print(f'Day of the year:    {yday_t1_integer}')

bearing_degrees = np.degrees(bearing) % 360
bearing_degrees = round(bearing_degrees, 1)  # Round to 2 decimal places
bearing_degrees = int(bearing_degrees)  # Convert to integer
print(f'Bearing (radians):  {bearing}')
print(f'Bearing (degrees):  {bearing_degrees}')

### Plot the sample covariates

In [ ]:
# Plot the covariates
fig, axs = plt.subplots(2, 1, figsize=(5, 9))

red = sample_spatial_covs.detach().cpu()[0, 3, :, :]
green = sample_spatial_covs.detach().cpu()[0, 2, :, :]
blue = sample_spatial_covs.detach().cpu()[0, 1, :, :]

# Assuming b4_tens, b3_tens, and b2_tens are your tensors
rgb_image = torch.stack([red, green, blue], dim=-1)
print(rgb_image.shape)
# Convert to NumPy
rgb_image_np = rgb_image.numpy()

# Normalize to the range [0, 1] for display
rgb_image_np = (rgb_image_np - rgb_image_np.min()) / (rgb_image_np.max() - rgb_image_np.min())

# Plot RGB
im1 = axs[0].imshow(rgb_image_np)
axs[0].set_title('Sentinel-2 RGB Image')

# Plot Slope
im2 = axs[1].imshow(sample_spatial_covs.detach().cpu().numpy()[0,13,:,:], cmap='viridis')
axs[1].set_title('Slope')
# fig.colorbar(im2, ax=axs[1])

filename_covs = f'{output_dir}/covs_yday{yday_t1_integer}_hour{hour_t1_integer}.png'
plt.tight_layout()
plt.savefig(filename_covs, dpi=300, bbox_inches='tight') # if we want to save the figure
plt.show()
plt.close()  # Close the figure to free memory

## Training loop

This code defines the main training loop for a single epoch. It iterates over batches from the training dataloader, moves the data to the correct device (e.g., CPU or GPU), calculates the loss, and performs backpropagation to update the model parameters. It also prints periodic updates of the current loss.

In [ ]:
train_loop = deepSSF_training_functions.train_loop
test_loop = deepSSF_training_functions.test_loop

## Loss function

In [ ]:
loss_fn = deepSSF_loss.negativeLogLikeLoss(reduction='mean')

## Early stopping

In [ ]:
early_stopping = deepSSF_early_stopping.EarlyStopping

# Train the model

Here we have the main training process that loops over multiple epochs. Each epoch involves:

1. Training the model on a training dataset.
2. Validating the model on a validation dataset to monitor its performance and adjust the learning rate (via scheduler).
3. Checking for early stopping conditions. If triggered, the best model weights are restored, and a test evaluation is performed.

Additionally, commented-out code at the end shows how you might visualise and save intermediate training results (such as predicted probability surfaces) for diagnostic or research purposes. The saved images can then be combined into an animation.

In [ ]:
print(f'Output directory: {output_dir}')
path_save_weights_pretraining = f'{output_dir}/checkpoint_deepSSF_model_pretraining.pt'
print(path_save_weights_pretraining)

## Training loop

In [ ]:
window_size = params_dict["image_dim"]  # Size of the input images

epochs = 150
train_losses = []  # Track training losses across epochs
val_losses = []   # Track validation losses across epochs
val_habitat_losses = []  # Track validation habitat losses across epochs
val_movement_losses = []  # Track validation movement losses across epochs
# Difference in loss between epochs
train_diff = []
val_diff = []
val_habitat_diff = []
val_movement_diff = []

# Initialize the parameter container using the parameters defined in 'params_dict'
params = deepSSF_model.ModelParams(params_dict)
# Create an instance of the ConvJointModel using the parameters,
# and move the model to the specified device (e.g., CPU or GPU)
model_pretraining = deepSSF_model.ConvJointModel(params).to(device)
# Print the model architecture
print(model_pretraining)

# Define the negative log-likelihood loss function with mean reduction
loss_fn = deepSSF_loss.negativeLogLikeLoss(reduction='mean') #, alpha=0.5

# Set the initial learning rates for each process
initial_learning_rate_movement = 1e-5
initial_learning_rate_habitat = 1e-4

# Create a combined optimiser for all movement-related parameters
# movement_params = list(model_pretraining.conv_movement.parameters()) + list(model_pretraining.fcn_movement_all.parameters())
movement_params = model_pretraining.fcn_movement_all.parameters()

# Define separate optimizers for each component
optimiser_movement = optim.Adam(movement_params, lr=initial_learning_rate_movement)
optimiser_habitat = optim.Adam(model_pretraining.conv_habitat.parameters(), lr=initial_learning_rate_habitat)

# Put optimisers into a tuple to call in the training loop
optimisers = (optimiser_movement, optimiser_habitat)

# Create separate schedulers for each optimizer
scheduler_movement = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimiser_movement, 'min', factor=0.1, patience=5)
scheduler_habitat = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimiser_habitat, 'min', factor=0.1, patience=5)

# Initialise early stopping 
early_stopping = deepSSF_early_stopping.EarlyStopping(patience=15, verbose=True, path=path_save_weights_pretraining)

# Create directory for saving training images
os.makedirs(f'{output_dir}/training_images', exist_ok=True)
os.makedirs(f'{output_dir}/loss_images', exist_ok=True)

for t in range(epochs):

    # Initialise variables to store during training
    train_loss = 0.0
    num_train_batches = len(pretraining_dataloader_train)

    val_loss = 0.0
    val_loss_habitat = 0.0
    val_loss_movement = 0.0
    num_batches = len(pretraining_dataloader_val)

    print(f"Epoch {t+1}\n-------------------------------")

    # Skip training in the first epoch, but still calculate losses
    skip_training = (t == 0)

    # 1. Run the training loop for one epoch using the training dataloader
    epoch_loss = train_loop(pretraining_dataloader_train, model_pretraining, loss_fn, optimisers, 
                            skip_epoch0_training=skip_training,
                            batch_size=batch_size)
    
    train_losses.append(epoch_loss.item())

    # 2. Evaluate model performance on the validation dataset
    model_pretraining.eval()  # Switch to evaluation mode for proper layer behavior
    with torch.no_grad():

        # Loop through each batch in the validation dataloader
        for x1, x2, x3, y, raster_transform in pretraining_dataloader_val:
            # Move data to the chosen device (CPU/GPU)
            x1 = x1.to(device)
            x2 = x2.to(device)
            x3 = x3.to(device)
            # y = y.to(device)

            # if isinstance(y, list):
            #     y = torch.stack(y)

            # Accumulate validation loss
            total_loss, habitat_loss, movement_loss = loss_fn(model_pretraining((x1, x2, x3)), y)
            val_loss += total_loss.detach()
            val_loss_habitat += habitat_loss.detach()
            val_loss_movement += movement_loss.detach()

    # # 3. Step the scheduler based on the validation loss (adjusts learning rate if needed)
    # scheduler.step(val_loss)

    # Step the movement scheduler
    scheduler_movement.step(val_loss_movement)

    # Step the habitat scheduler regardless of the movement weight
    scheduler_habitat.step(val_loss_habitat)

    # 4. Compute the average validation loss and print it, along with the current learning rate
    val_loss /= num_batches
    val_loss_habitat /= num_batches
    val_loss_movement /= num_batches

    print(f"Avg validation loss:            {val_loss:>15f}")
    print(f"Avg validation habitat loss:    {val_loss_habitat:>15f}")
    print(f"Avg validation movement loss:   {val_loss_movement:>15f}")
    print(f"Movement learning rate:         {scheduler_movement.get_last_lr()}")
    print(f"Habitat learning rate:          {scheduler_habitat.get_last_lr()}")

    # 5. Track the validation loss for plotting or monitoring
    val_losses.append(val_loss.item())
    val_habitat_losses.append(val_loss_habitat.item())
    val_movement_losses.append(val_loss_movement.item())

    # Memory management - add after validation but before early stopping check
    # torch.mps.empty_cache()
    # gc.collect()

    # 6. Early stopping: if no improvement in validation loss for a set patience, stop training
    early_stopping(val_loss, model_pretraining)
    if early_stopping.early_stop:
        print("Early stopping")
        # Restore the best model weights saved by EarlyStopping
        model_pretraining.load_state_dict(torch.load(path_save_weights_pretraining, weights_only=True, map_location=device))
        test_loop(pretraining_dataloader_test, model_pretraining, loss_fn)  # Evaluate on test set once training stops
        break
    else:
        model_pretraining.eval()
        print("\n")

    torch.cuda.empty_cache()


    # ----------------------------------------------------
    # The following code demonstrates how
    # to optionally visualize or save intermediate results
    # (e.g., habitat probability surface, movement probability,
    # and next-step probability surfaces).

    # uncomment the code all in one go to run it (it should be inside the training loop)
    # ----------------------------------------------------

    # Extract training and validation losses for plotting

    # Convert the list of tensors to a single tensor
    train_losses_np = torch.tensor(train_losses).detach().cpu().numpy()
    val_losses_np = torch.tensor(val_losses).detach().cpu().numpy()
    val_habitat_losses_np = torch.tensor(val_habitat_losses).detach().cpu().numpy()
    val_movement_losses_np = torch.tensor(val_movement_losses).detach().cpu().numpy()

    # Get the difference in losses between epochs
    train_diff.append(train_losses_np[t] - train_losses_np[t-1])
    val_diff.append(val_losses_np[t] - val_losses_np[t-1])
    val_habitat_diff.append(val_habitat_losses_np[t] - val_habitat_losses_np[t-1])
    val_movement_diff.append(val_movement_losses_np[t] - val_movement_losses_np[t-1])

    # Number of epochs
    n_epochs = len(val_losses)

    # -----------------------------------------------------------
    # 1. Retrieve a single test example (covariates and labels)
    #    at the specified 'iteration_index' from the test dataset
    # -----------------------------------------------------------
    x1, x2, x3, labels, raster_transform = pretraining_dataloader_train.dataset[int(iteration_index)]

    # -----------------------------------------------------------
    # 2. Add a batch dimension and move tensors to the device
    #    for model inference
    # -----------------------------------------------------------
    x1 = x1.unsqueeze(0).to(device)
    x2 = x2.unsqueeze(0).to(device)
    x3 = x3.unsqueeze(0).to(device)

    # -----------------------------------------------------------
    # 3. Run the model on the single test example
    # -----------------------------------------------------------
    test = model_pretraining((x1, x2, x3))

    # -----------------------------------------------------------
    # 4. Extract habitat and movement outputs;
    #    convert them to NumPy arrays for visualization
    # -----------------------------------------------------------
    hab_density = test.detach().cpu().numpy()[0, :, :, 0]
    movement_density = test.detach().cpu().numpy()[0, :, :, 1]

    # -----------------------------------------------------------
    # 5. Generate masks to exclude certain border cells for
    #    color scale reasons (setting them to -inf).
    # -----------------------------------------------------------
    x_mask = np.ones_like(hab_density)
    y_mask = np.ones_like(hab_density)

    # Mask out a few columns (0-2 and 98-end) and rows (0-2 and 98-end)
    x_mask[:, :n_conv_layers] = -np.inf
    x_mask[:, window_size-n_conv_layers:] = -np.inf
    y_mask[:n_conv_layers, :] = -np.inf
    y_mask[window_size-n_conv_layers:, :] = -np.inf

    # Apply the masks to habitat density
    hab_density_mask = hab_density * x_mask * y_mask

    # Combine habitat and movement densities to represent
    # next-step probability
    step_density = hab_density + movement_density
    step_density_mask = step_density * x_mask * y_mask

    # Plot the covariates
    fig, axs = plt.subplots(2, 2, figsize=(9, 7.5))

    # # Plot NDVI
    # im1 = axs[0, 0].imshow(ndvi_natural.numpy(), cmap='viridis')
    # axs[0, 0].set_title('NDVI')
    # fig.colorbar(im1, ax=axs[0, 0])

    # Plot Training and Validation Loss
    axs[0, 0].plot(range(n_epochs), train_losses_np, label='Training Loss', color='blue')
    axs[0, 0].plot(range(n_epochs), val_losses_np, label='Validation Loss', color='red')
    # to show the habitat and movement losses, uncomment the following lines (on quite different scales)
    # axs[0, 0].plot(range(n_epochs), val_habitat_losses_np, label='Validation Habitat Loss', color='green')
    # axs[0, 0].plot(range(n_epochs), val_movement_losses_np, label='Validation Movement Loss', color='orange')
    axs[0, 0].set_xlim(0, n_epochs)
    axs[0, 0].set_title('Training and validation loss')
    axs[0, 0].set_xlabel('Epoch')
    axs[0, 0].set_ylabel('Loss')
    axs[0, 0].legend()

    # Plot habitat selection log-probability
    im2 = axs[0, 1].imshow(hab_density_mask, cmap='viridis')
    axs[0, 1].set_title('Habitat log-probability')
    fig.colorbar(im2, ax=axs[0, 1])

    # Plot movement log-probability
    im3 = axs[1, 0].imshow(movement_density, cmap='viridis')
    axs[1, 0].set_title('Movement log-probability')
    fig.colorbar(im3, ax=axs[1, 0])

    # Plot next-step log-probability
    im4 = axs[1, 1].imshow(step_density_mask, cmap='viridis')
    axs[1, 1].set_title('Next-step log-probability')
    fig.colorbar(im4, ax=axs[1, 1])

    filename_covs = f'{output_dir}/training_images/training_epoch_index{t}_yday{yday_t1_integer}_hour{hour_t1_integer}_bearing{bearing_degrees}.png'
    plt.tight_layout()
    plt.savefig(filename_covs, dpi=150) # creates inconsistent image sizes >>> , bbox_inches='tight'
    # plt.show()
    plt.close()  # Close the figure to free memory

    # Plot the difference in the loss of each component between epochs
    filename_diff = f'{output_dir}/loss_images/training_diff_epoch_index{t}_yday{yday_t1_integer}_hour{hour_t1_integer}_bearing{bearing_degrees}.png'
    plt.axhline(y=0, color='black', linestyle='--', label='Null Probability')  # null probs
    # plt.plot(range(n_epochs), train_diff, label='Training Loss Difference', color='blue')
    # plt.plot(range(n_epochs), val_diff, label='Validation Loss Difference', color='red')
    plt.plot(range(n_epochs), val_habitat_diff, label='Validation Habitat Loss Difference', color='green')
    plt.plot(range(n_epochs), val_movement_diff, label='Validation Movement Loss Difference', color='orange')
    plt.xlim(0, n_epochs)
    plt.title('Habitat and movement loss difference')
    plt.xlabel('Epoch')
    plt.ylabel('Loss difference')
    plt.legend()
    plt.savefig(filename_diff, dpi=150) 
    # plt.show()
    plt.close()  # Close the figure to free memory

print("Done!")

### Make a GIF of the training images

First, here's a function to call to make a gif from a given directory.

In [ ]:
# Example sorting by the epoch number
def extract_index(filename):
    # Extract the epoch number from the filename
    # Adjust the extraction based on your naming pattern
    import re
    match = re.search(r'index(\d+)_', filename)
    if match:
        return int(match.group(1))
    return 0

def create_gif(image_folder, output_filename, fps=10):
    """
    Creates a GIF from a sequence of images in a folder.

    Parameters:
    - image_folder: Path to the folder containing images
    - output_filename: Name of the output GIF file
    - duration: Duration of each frame in seconds
    """
    # Get all png files in the specified folder, sorted by name
    images = sorted(glob.glob(os.path.join(image_folder, '*.png')), key=extract_index)

    # Check if any images were found
    if not images:
        print(f"No images found in {image_folder}")
        return

    # Read all images
    frames = [imageio.imread(image) for image in images]

    # Create the animation file
    if output_filename.endswith('.gif'):
        print(f"GIF saved as {output_filename}")
        # Save as GIF
        imageio.mimsave(output_filename, frames, fps=fps, loop=0, 
                        quantizer='nq')  # Neural-Net quantizer (better quality)
        display(Image(filename=output_filename))
    else:
        imageio.mimsave(output_filename, frames, fps=fps, quality=8)
        print(f"Animation saved as: {output_filename}")

    print(f"Animation created successfully: {output_filename}")


## Create training GIF

In [ ]:
# Path to your images
image_folder =  f'{output_dir}/training_images'
# Output GIF filename
output_filename = f'{output_dir}/training_gif_yday{yday_t1_integer}_hour{hour_t1_integer}_bearing{bearing_degrees}.gif'
# Create the GIF
create_gif(image_folder, output_filename, fps=10)

# Output MP4 filename
output_filename = f'{output_dir}/training_mp4_yday{yday_t1_integer}_hour{hour_t1_integer}_bearing{bearing_degrees}.mp4'
# Create the MP4
create_gif(image_folder, output_filename, fps=10)

## Create loss GIF

In [ ]:
# Path to your images
image_folder =  f'{output_dir}/loss_images'
# Output GIF filename
output_filename = f'{output_dir}/loss_gif.gif'
# Create the GIF
create_gif(image_folder, output_filename, fps=10)

In [ ]:
# to look at the parameters (weights and biases) of the model
# print(model_pretraining.state_dict())

# Loading in previous models

As we've trained the model, the model parameters are already stored in the `model` object. 

The model parameters that are being loaded must match the model object that has been defined above. If the model object has changed, the model parameters will not be able to be loaded.

We load the model object in the output directory defined at the top of the script.

In [ ]:
# Print where the model weights are saved
print(os.path.join(base_path, path_save_weights_pretraining))

### If loading a previously trained model

If we want to load a model from a different directory, we can specify it manually here, but the architecture must match.

In [ ]:
# to load previously saved weights
# path_save_weights_manual = f'{output_dir}/checkpoint_deepSSF_buffalo_pretraining.pt'

# model_pretraining.load_state_dict(torch.load(path_save_weights_manual,
#                                  weights_only=True,
#                                  map_location=torch.device('cpu')))

# View model outputs

## Create a directory to save model outputs

### Save the validation loss as a dataframe

In [ ]:
# Directory for saving the loss dataframe
filename_loss_csv = f'{output_dir}/deepSSF_val_loss.csv'

# Check if val_losses is defined (which means a model has been trained in this session)
try:

    # Convert the list of tensors to a single tensor
    val_losses_tensor = torch.tensor(val_losses)

    print("val_losses has been defined - storing as csv\n")

    # Number of epochs
    n_epochs = len(val_losses)
    print(f'Number of epochs: {n_epochs}')

    val_losses_df = pd.DataFrame({
        "epoch": range(1, n_epochs + 1),
        "val_losses": val_losses_tensor.detach().cpu().numpy()
    })

    print(val_losses_df.head())

    # Save the validation losses to a CSV file
    val_losses_df.to_csv(filename_loss_csv, index=False)

# if val_losses hasn't been defined (for if you are loading model weights from a saved object)
except NameError:

    # This code runs if val_losses is not defined
    print("val_losses has not been defined - loading from saved csv\n")
    # Initialize it with a default value

    # Read the val_losses csv file
    val_losses_df = pd.read_csv(filename_loss_csv)
    print(val_losses_df.head())

    # Number of epochs
    n_epochs = len(val_losses_df)
    print(f'\nNumber of epochs: {n_epochs}')


In [ ]:
# Directory for saving the loss dataframe
filename_train_loss_csv = f'{output_dir}/deepSSF_train_loss.csv'

# Check if train_losses is defined (which means a model has been trained in this session)
try:

    # Convert the list of tensors to a single tensor
    train_losses_tensor = torch.tensor(train_losses)

    print("train_losses has been defined - storing as csv\n")

    train_losses_df = pd.DataFrame({
        "epoch": np.linspace(1, n_epochs, len(train_losses)),
        "train_losses": train_losses_tensor.detach().cpu().numpy()
    })

    print(train_losses_df.head)

    # Save the train losses to a CSV file
    train_losses_df.to_csv(filename_train_loss_csv, index=False)

# if train_losses hasn't been defined (for if you are loading model weights from a saved object)
except NameError:

    # This code runs if train_losses is not defined
    print("train_losses has not been defined - loading from saved csv\n")
    # Initialize it with a default value

    # Read the train_losses csv file
    train_losses_df = pd.read_csv(filename_train_loss_csv)
    print(train_losses_df.head())


### Plot the validation loss

In [ ]:
# Directory for saving the loss plots
filename_loss = f'{output_dir}/val_loss.png'

# Plot the validation losses
plt.plot(train_losses_df['epoch'], train_losses_df['train_losses'], label='Training Loss', color='blue')  # Plot training loss in blue
plt.plot(val_losses_df['epoch'], val_losses_df['val_losses'], label='Validation Loss', color='red')  # Plot validation loss in red
plt.title('Validation Losses')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()  # Show legend to distinguish lines
plt.savefig(filename_loss, dpi=300, bbox_inches='tight')
plt.show()

# Test model

Take some random samples from the test dataset and generate predictions for them. We loop through the samples (which are shuffled randomly), make predictions, and plot the results.

In [ ]:
# 1. Set the model in evaluation mode
model_pretraining.eval()

# Loop over samples in the validation dataset
for i in range(0, 5):

  sample_number = np.random.randint(0, len(pretraining_dataloader_test))
  print(f'Sample number: {sample_number}')

  # Display image and label
  x1, x2, x3, labels, raster_transform = pretraining_dataloader_test.dataset[sample_number]

  # Add a batch dimension
  x1 = x1.unsqueeze(0).cpu()
  x2 = x2.unsqueeze(0).cpu()
  x3 = x3.unsqueeze(0).cpu()

  # Pull out the scalars
  hour_t1_sin1 = x2.detach().numpy()[0,0]
  hour_t1_cos1 = x2.detach().numpy()[0,1]
  yday_t1_sin1 = x2.detach().numpy()[0,2]
  yday_t1_cos1 = x2.detach().numpy()[0,3]
  bearing = x3.detach().numpy()[0,0]

  # Recover the hour
  hour_t1 = deepSSF_utils.recover_hour(hour_t1_sin1, hour_t1_cos1)
  hour_t1_integer = int(hour_t1)  # Convert to integer
  print(f'Hour:                        {hour_t1_integer}')

  # Recover the day of the year
  yday_t1 = deepSSF_utils.recover_yday(yday_t1_sin1, yday_t1_cos1)
  yday_t1_integer = int(yday_t1)  # Convert to integer
  print(f'Day of the year:             {yday_t1_integer}')

  # Recover the bearing
  bearing_degrees = np.degrees(bearing) % 360
  bearing_degrees = round(bearing_degrees, 1)  # Round to 2 decimal places
  bearing_degrees = int(bearing_degrees)  # Convert to integer
  print(f'Bearing (radians):           {bearing}')
  print(f'Bearing (degrees):           {bearing_degrees}')

  # Pull out the RGB layers for plotting
  blue_layer = x1.detach().cpu().numpy()[0,1,:,:]
  green_layer = x1.detach().cpu().numpy()[0,2,:,:]
  red_layer = x1.detach().cpu().numpy()[0,3,:,:]

  # Stack the RGB layers
  rgb_image_np = np.stack([red_layer, green_layer, blue_layer], axis=-1)

  # Normalize to the range [0, 1] for display
  rgb_image_np = (rgb_image_np - rgb_image_np.min()) / (rgb_image_np.max() - rgb_image_np.min())

  # Find the coordinates of the element that is 1
  coordinates = labels

  # Extract the coordinates
  row, column = coordinates[0], coordinates[1]
  print(f"Next step is (row, column):  ({row}, {column})")


  # -------------------------------------------------------------------------
  # Run the model on the input data
  # -------------------------------------------------------------------------

  # Move input tensors to the GPU if available
  x1 = x1.to(device)
  x2 = x2.to(device)
  x3 = x3.to(device)

  test = model_pretraining((x1, x2, x3))
  # print(test.shape)

  # Extract and exponentiate the habitat density channel
  hab_density = test.detach().cpu().numpy()[0, :, :, 0]
  hab_density_exp = np.exp(hab_density)
  # print(np.sum(hab_density_exp))  # Debug: check the sum of exponentiated values

  # Create masks to remove unwanted edge cells from visualization
  #    (setting them to -∞ affects the color scale in plots)
  x_mask = np.ones_like(hab_density)
  y_mask = np.ones_like(hab_density)

  # mask out cells on the edges that affect the colour scale
  x_mask[:, :n_conv_layers] = -np.inf
  x_mask[:, window_size-n_conv_layers:] = -np.inf
  y_mask[:n_conv_layers, :] = -np.inf
  y_mask[window_size-n_conv_layers:, :] = -np.inf

  next_step_mask = np.ones_like(hab_density)
  next_step_mask[row, column] = -np.inf

  # Apply the masks to the habitat density (log scale) and exponentiated version
  hab_density_mask = hab_density * x_mask * y_mask * next_step_mask
  hab_density_exp_mask = hab_density_exp * x_mask * y_mask * next_step_mask

  # Extract and exponentiate the movement density channel
  move_density = test.detach().cpu().numpy()[0,:,:,1]
  move_density_exp = np.exp(move_density)

  # Apply the same masking strategy to movement densities
  move_density_mask = move_density * x_mask * y_mask * next_step_mask
  move_density_exp_mask = move_density_exp * x_mask * y_mask * next_step_mask

  # Compute the next-step density by adding habitat + movement (log-space)
  step_density = test[0, :, :, 0] + test[0, :, :, 1]
  step_density = step_density.detach().cpu().numpy()
  step_density_exp = np.exp(step_density)

  # Apply masks to the step densities (log and exponentiated)
  step_density_mask = step_density * x_mask * y_mask * next_step_mask
  step_density_exp_mask = step_density_exp * x_mask * y_mask * next_step_mask

  # -------------------------------------------------------------------------
  # Plot the RGB image, slope, habitat selection, and movement density
  #   Change the panels to visualize different layers
  # -------------------------------------------------------------------------
  fig, axs = plt.subplots(2, 2, figsize=(10, 10))

  # Plot RGB
  im1 = axs[0, 0].imshow(rgb_image_np)
  axs[0, 0].set_title('Sentinel-2 RGB')

  # Plot slope
  im2 = axs[0, 1].imshow(x1.detach().cpu().numpy()[0,13,:,:], cmap='viridis')
  axs[0, 1].set_title('Slope')
  fig.colorbar(im2, ax=axs[0, 1], shrink=0.7)

  # Plot habitat selection
  im3 = axs[1, 0].imshow(hab_density_mask, cmap='viridis')
  axs[1, 0].set_title('Habitat selection log-probability')
  fig.colorbar(im3, ax=axs[1, 0], shrink=0.7)

  # # Movement density (change the axis and uncomment one of the other panels)
  # im3 = axs[1, 0].imshow(move_density_mask, cmap='viridis')
  # axs[1, 0].set_title('Movement log-probability')
  # fig.colorbar(im3, ax=axs[0, 1], shrink=0.7)

  # Next-step probability
  im4 = axs[1, 1].imshow(step_density_mask, cmap='viridis')
  axs[1, 1].set_title('Next-step log-probability')
  fig.colorbar(im4, ax=axs[1, 1], shrink=0.7)

  # Save the figure
  filename_covs = f'{output_dir}/deepSSF_S2_slope_id_yday{yday_t1_integer}_hour{hour_t1_integer}_bearing{bearing_degrees}_next_r{row}_c{column}.png'
  plt.tight_layout()
  plt.savefig(filename_covs, dpi=600, bbox_inches='tight')
  plt.show()
  plt.close()  # Close the figure to free memory


In [ ]:
x1, x2, x3, labels, raster_transform = pretraining_dataloader_test.dataset[sample_number]

# Add a batch dimension and move tensors to the device
x1 = x1.unsqueeze(0).to(device)
x2 = x2.unsqueeze(0).to(device)
x3 = x3.unsqueeze(0).to(device)

# Extracting convolution layer outputs

In the convolutional blocks, each convolutional layer learns a set of **filters** (kernels) that extract different features from the input data. In the habitat selection subnetwork, the convolution filters (and their associated bias parameters - not shown below) are the only parameters that are trained, and it is the filters that transform the set of input covariates into the habitat selection probabilities. They do this by maximising features of the inputs that correlate with observed next-steps.

For each convolutional layer, there are typically a number of filters. For the habitat selection subnetwork, we used 4 filters in the first two layers, and a single filter in the last layer. Each of these filters has a number of **channels** which correspond one-to-one with the input layers. The outputs of the filter channels are then combined to produce a feature map, with a single feature map produced for each filter. In successive layers, the feature maps become the input layers, and the filters operate on these layers. Because there are multiple filters in ech layer, they can 'specialise' in extracting different features from the input layers.

By visualizing and inspecting these filters, and the corresponding feature maps, we can:

- Gain interpretability: Understand what kind of features the network is detecting—e.g., edges, shapes, or textures.
- Debug: Check if the filters have meaningful patterns or if something went wrong (e.g., all zeros or random noise).
- Compare layers: See how early layers often learn low-level patterns while deeper layers learn more abstract features.

We will first set up some activation hooks for storing the feature maps. Activation hooks are placed at certain points within the model's forward pass and store intermediate results. We will also extract the convolution filters (which are weights of the model and as such don't require hooks - we can access them directly).

We will then run the sample covariates through the model and extract the feature maps from the habitat selection convolutional block, and plot them along with the covariates and convolution filters.

Note that there are also ReLU activation functions in the convolutional blocks, which are not shown below. These are applied to the feature maps, and set all negative values to zero. They are not learned parameters, but are part of the forward pass of the model.


### Create scalar grids for plotting

Using the `Scalar_to_Grid_Block` class from the `deepSSF_model` script, we can convert the scalar covariates into grids for plotting.

In [ ]:
# Create an instance of the scalar-to-grid block using model parameters
scalar_to_grid_block = deepSSF_model.Scalar_to_Grid_Block(params)

# Convert scalars into spatial grid representation
scalar_maps = scalar_to_grid_block(x2)
print(scalar_maps.shape)  # Check the shape of the generated spatial maps

## Convolutional layer 1

### Activation hook

In [ ]:
# -----------------------------------------------------------
# Create a dictionary to store activation outputs
# -----------------------------------------------------------
activation = {}

def get_activation(name):
    """
    Returns a hook function that can be registered on a layer
    to capture its output (i.e., feature maps) after the forward pass.

    Args:
        name (str): The key under which the activation is stored in the 'activation' dict.
    """
    def hook(model, input, output):
        # Detach and save the layer's output in the dictionary
        activation[name] = output.detach()
    return hook

# -----------------------------------------------------------
# Register a forward hook on the first convolution layer
#    in the model's 'conv_habitat' block
# -----------------------------------------------------------
model_pretraining.conv_habitat.conv2d[0].register_forward_hook(get_activation("hab_conv1"))

# -----------------------------------------------------------
# Perform a forward pass through the model with the desired input
#    The feature maps from the hooked layer will be stored in 'activation'
# -----------------------------------------------------------
out = model_pretraining((x1, x2, x3))  # e.g., model((spatial_data_x, scalars_to_grid, bearing_x))

# -----------------------------------------------------------
# Retrieve the captured feature maps from the dictionary
#    and move them to the CPU for inspection
# -----------------------------------------------------------
feat_maps1 = activation["hab_conv1"].cpu()
print("Feature map shape:", feat_maps1.shape)
# Typically shape: (batch_size, out_channels, height, width)

# -----------------------------------------------------------
# Visualize the feature maps for the first sample in the batch
# -----------------------------------------------------------
feat_maps1_sample = feat_maps1[0]  # Shape: (out_channels, H, W)
num_maps1 = feat_maps1_sample.shape[0]
print("Number of feature maps:", num_maps1)



### Stack spatial and scalar (as grid) covariates

For plotting. Also create a vector of names to index over.

In [ ]:
covariate_stack = torch.cat([x1, scalar_maps], dim=1)
print(covariate_stack.shape)

covariate_names = ['S2 B1',
                   'S2 B2',
                   'S2 B3',
                   'S2 B4',
                   'S2 B5',
                   'S2 B6',
                   'S2 B7',
                   'S2 B8',
                   'S2 B8a',
                   'S2 B9',
                   'S2 B11',
                   'S2 B12',
                   'Elevation',
                   'Slope',
                   'Hour sin1',
                   'Hour cos1',
                   'yday sin1',
                   'yday cos1',
                   'dt_hour']

### Extract filters and plot

In [ ]:
# -------------------------------------------------------------------------
# Check or print the convolution layer in conv_habitat (for debugging)
# -------------------------------------------------------------------------
print(model_pretraining.conv_habitat.conv2d)

# -------------------------------------------------------------------------
# Set the model to evaluation mode (disables dropout, etc.)
# -------------------------------------------------------------------------
model_pretraining.eval()

# -------------------------------------------------------------------------
# Extract the weights (filters) from the first convolution layer in conv_habitat
# -------------------------------------------------------------------------
filters_c1 = model_pretraining.conv_habitat.conv2d[0].weight.data.clone().cpu()
print("Filters shape:", filters_c1.shape)
# Typically (out_channels, in_channels, kernel_height, kernel_width)

# -------------------------------------------------------------------------
# Visualize each filter’s first channel in a grid of subplots
# -------------------------------------------------------------------------
num_filters_c1 = filters_c1.shape[1]
print(num_filters_c1)

for z in range(num_maps1):

    fig, axes = plt.subplots(2, num_filters_c1, figsize=(2*num_filters_c1, 4))
    for i in range(num_filters_c1):

        # Add the covariates as the first row of subplots
        axes[0,i].imshow(covariate_stack[0, i].detach().cpu().numpy(), cmap='viridis')
        axes[0,i].axis('off')
        axes[0,i].set_title(f'{covariate_names[i]}')
        if i > x1.shape[1] - 1:
            im1 = axes[0,i].imshow(covariate_stack[0, i].detach().cpu().numpy(), cmap='viridis')
            im1.set_clim(-1, 1)
            axes[0,i].text(scalar_maps.shape[2] // 2, scalar_maps.shape[3] // 2,
                f'Value: {round(x2[0, i-x1.shape[1]].item(), 2)}',
                ha='center', va='center', color='white', fontsize=12)

        kernel = filters_c1[z, i, :, :]  # Show the first input channel
        im = axes[1,i].imshow(kernel, cmap='viridis', vmin=-0.5, vmax=0.5)
        axes[1,i].axis('off')
        axes[1,i].set_title(f'Layer 1, Filter {z+1}')
        # Annotate each cell with the numeric value
        for (j, k), val in np.ndenumerate(kernel):
            axes[1,i].text(k, j, f'{val:.2f}', ha='center', va='center', color='white')

    plt.tight_layout()
    plt.savefig(f'{output_dir}/conv_layer1_filters{z}_{today_date}.png', dpi=600, bbox_inches='tight')
    plt.show()


    # -----------------------------------------------------------
    # Loop over each feature map channel and save them as images.
    #    Multiply by x_mask * y_mask if you need to mask out edges.
    # -----------------------------------------------------------

    plt.figure()
    plt.imshow(feat_maps1_sample[z].numpy() * x_mask * y_mask, cmap='viridis')
    plt.title(f"Layer 1, Feature Map {z+1}")
    # Hide axis if you prefer: plt.axis('off')
    plt.savefig(f'{output_dir}/conv_layer1_feature_map{z}_{today_date}.png', dpi=600, bbox_inches='tight')
    plt.show()



## Convolutional layer 2

### Activation hook

In [ ]:
# -----------------------------------------------------------
# Register a forward hook on the second convolution layer
#    in the model's 'conv_habitat' block
# -----------------------------------------------------------
model_pretraining.conv_habitat.conv2d[2].register_forward_hook(get_activation("hab_conv2"))

# -----------------------------------------------------------
# Perform a forward pass through the model with the desired input
#    The feature maps from the hooked layer will be stored in 'activation'
# -----------------------------------------------------------
out = model_pretraining((x1, x2, x3))  # e.g., model((spatial_data_x, scalars_to_grid, bearing_x))

# -----------------------------------------------------------
# Retrieve the captured feature maps from the dictionary
#    and move them to the CPU for inspection
# -----------------------------------------------------------
feat_maps2 = activation["hab_conv2"].cpu()
print("Feature map shape:", feat_maps2.shape)
# Typically shape: (batch_size, out_channels, height, width)

# -----------------------------------------------------------
# Visualize the feature maps for the first sample in the batch
# -----------------------------------------------------------
feat_maps2_sample = feat_maps2[0]  # Shape: (out_channels, H, W)
num_maps2 = feat_maps2_sample.shape[0]
print("Number of feature maps:", num_maps2)



### Extract filters and plot

In [ ]:
# -------------------------------------------------------------------------
# Extract the weights (filters) from the second convolution layer in conv_habitat
# -------------------------------------------------------------------------
filters_c2 = model_pretraining.conv_habitat.conv2d[2].weight.data.clone().cpu()
print("Filters shape:", filters_c2.shape)
# Typically (out_channels, in_channels, kernel_height, kernel_width)

# -------------------------------------------------------------------------
# Visualize each filter’s first channel in a grid of subplots
# -------------------------------------------------------------------------
num_filters_c2 = filters_c2.shape[1]
print(num_filters_c2)

for z in range(num_maps2):

    fig, axes = plt.subplots(2, num_filters_c2, figsize=(2*num_filters_c2, 4))
    for i in range(num_filters_c2):

        # Add the covariates as the first row of subplots
        axes[0,i].imshow(feat_maps1_sample[i].numpy() * x_mask * y_mask, cmap='viridis')
        axes[0,i].axis('off')
        axes[0,i].set_title(f"Layer 1, Map {z+1}")

        # if i > 3:
        #     im1 = axes[0,i].imshow(covariate_stack[0, i].detach().cpu().numpy(), cmap='viridis')
        #     im1.set_clim(-1, 1)
        #     axes[0,i].text(scalar_maps.shape[2] // 2, scalar_maps.shape[3] // 2,
        #         f'Value: {round(x2[0, i-4].item(), 2)}',
        #         ha='center', va='center', color='white', fontsize=12)

        kernel = filters_c2[z, i, :, :]  # Show the first input channel
        im = axes[1,i].imshow(kernel, cmap='viridis', vmin=-0.5, vmax=0.5)
        axes[1,i].axis('off')
        axes[1,i].set_title(f'Layer 2, Filter {z+1}')
        # Annotate each cell with the numeric value
        for (j, k), val in np.ndenumerate(kernel):
            axes[1,i].text(k, j, f'{val:.2f}', ha='center', va='center', color='white')

    plt.tight_layout()
    plt.savefig(f'{output_dir}/conv_layer2_filters{z}_{today_date}.png', dpi=600, bbox_inches='tight')
    plt.show()


    # -----------------------------------------------------------
    # 6. Loop over each feature map channel and save them as images.
    #    Multiply by x_mask * y_mask if you need to mask out edges.
    # -----------------------------------------------------------

    plt.figure()
    plt.imshow(feat_maps2_sample[z].numpy() * x_mask * y_mask, cmap='viridis')
    plt.title(f"Layer 2, Feature Map {z+1}")
    # Hide axis if you prefer: plt.axis('off')
    plt.savefig(f'{output_dir}/conv_layer2_feature_map{z}_{today_date}.png', dpi=600, bbox_inches='tight')
    plt.show()



## Convolutional layer 3

### Activation hook

In [ ]:
# -----------------------------------------------------------
# Register a forward hook on the third convolution layer
#    in the model's 'conv_habitat' block
# -----------------------------------------------------------
model_pretraining.conv_habitat.conv2d[4].register_forward_hook(get_activation("hab_conv3"))

# -----------------------------------------------------------
# Perform a forward pass through the model with the desired input
#    The feature maps from the hooked layer will be stored in 'activation'
# -----------------------------------------------------------
out = model_pretraining((x1, x2, x3))  # e.g., model((spatial_data_x, scalars_to_grid, bearing_x))

# -----------------------------------------------------------
# Retrieve the captured feature maps from the dictionary
#    and move them to the CPU for inspection
# -----------------------------------------------------------
feat_maps3 = activation["hab_conv3"].cpu()
print("Feature map shape:", feat_maps3.shape)
# Typically shape: (batch_size, out_channels, height, width)

# -----------------------------------------------------------
# Visualize the feature maps for the first sample in the batch
# -----------------------------------------------------------
feat_maps3_sample = feat_maps3[0]  # Shape: (out_channels, H, W)
num_maps3 = feat_maps3_sample.shape[0]
print("Number of feature maps:", num_maps3)



### Extract filters and plot

In [ ]:
# -------------------------------------------------------------------------
# Extract the weights (filters) from the second convolution layer in conv_habitat
# -------------------------------------------------------------------------
filters_c3 = model_pretraining.conv_habitat.conv2d[4].weight.data.clone().cpu()
print("Filters shape:", filters_c3.shape)
# Typically (out_channels, in_channels, kernel_height, kernel_width)

# -------------------------------------------------------------------------
# Visualize each filter’s first channel in a grid of subplots
# -------------------------------------------------------------------------
num_filters_c3 = filters_c3.shape[1]
print(num_filters_c3)

for z in range(num_maps3):

    fig, axes = plt.subplots(2, num_filters_c3, figsize=(2*num_filters_c3, 4))
    for i in range(num_filters_c3):

        # Add the covariates as the first row of subplots
        axes[0,i].imshow(feat_maps2_sample[i].numpy() * x_mask * y_mask, cmap='viridis')
        axes[0,i].axis('off')
        axes[0,i].set_title(f"Layer 2, Map {z+1}")


        kernel = filters_c3[z, i, :, :]  # Show the first input channel
        im = axes[1,i].imshow(kernel, cmap='viridis', vmin=-0.5, vmax=0.5)
        axes[1,i].axis('off')
        axes[1,i].set_title(f'Layer 3, Filter {z+1}')
        # Annotate each cell with the numeric value
        for (j, k), val in np.ndenumerate(kernel):
            axes[1,i].text(k, j, f'{val:.2f}', ha='center', va='center', color='white')

    plt.tight_layout()
    plt.savefig(f'{output_dir}/conv_layer3_filters{z}_{today_date}.png', dpi=600, bbox_inches='tight')
    plt.show()


    # -----------------------------------------------------------
    # 6. Loop over each feature map channel and save them as images.
    #    Multiply by x_mask * y_mask if you need to mask out edges.
    # -----------------------------------------------------------

    plt.figure()
    plt.imshow(feat_maps3_sample[z].numpy() * x_mask * y_mask, cmap='viridis')
    plt.title(f"Habitat selection log probability")
    # Hide axis if you prefer: plt.axis('off')
    plt.savefig(f'{output_dir}/conv_layer3_feature_map{z}_{today_date}.png', dpi=600, bbox_inches='tight')
    plt.show()



# Checking estimated movement parameters

Similarly to the convolutional layers, we can set hooks to extract the predicted movement parameters from the model, and assess how variable that is across samples.

In [ ]:
# -------------------------------------------------------------------------
# Create a list to store the intermediate output from the fully connected
#    movement sub-network (fcn_movement_all)
# -------------------------------------------------------------------------
intermediate_output = []

def hook(module, input, output):
    """
    Hook function that captures the output of the specified layer
    (fcn_movement_all) during the forward pass.
    """
    intermediate_output.append(output)

# -------------------------------------------------------------------------
# Register the forward hook on 'fcn_movement_all', so its outputs
#    are recorded every time the model does a forward pass.
# -------------------------------------------------------------------------
hook_handle = model_pretraining.fcn_movement_all.register_forward_hook(hook)

# -------------------------------------------------------------------------
# Perform a forward pass with the model in evaluation mode,
#    disabling gradient computation.
# -------------------------------------------------------------------------
model_pretraining.eval()
with torch.no_grad():
    final_output = model_pretraining((x1, x2, x3))

# -------------------------------------------------------------------------
# Inspect the captured intermediate output
#    'intermediate_output[0]' corresponds to the first (and only) forward pass.
# -------------------------------------------------------------------------
print("Intermediate output shape:", intermediate_output[0].shape)
print("Intermediate output values:", intermediate_output[0][0])

# -------------------------------------------------------------------------
# Remove the hook to avoid repeated capturing in subsequent passes
# -------------------------------------------------------------------------
hook_handle.remove()

# -------------------------------------------------------------------------
# Unpack the parameters from the FCN output (assumes a specific ordering)
# -------------------------------------------------------------------------
gamma_shape1, gamma_scale1, gamma_weight1, \
gamma_shape2, gamma_scale2, gamma_weight2, \
vonmises_mu1, vonmises_kappa1, vonmises_weight1, \
vonmises_mu2, vonmises_kappa2, vonmises_weight2 = intermediate_output[0][0].cpu()

# -------------------------------------------------------------------------
# Convert parameters from log-space (if applicable) and print them
#    Gamma and von Mises parameters
# -------------------------------------------------------------------------
# --- Gamma #1 ---
print("Gamma shape 1:", torch.exp(gamma_shape1))
print("Gamma scale 1:", torch.exp(gamma_scale1))
print("Gamma weight 1:",
      torch.exp(gamma_weight1) / (torch.exp(gamma_weight1) + torch.exp(gamma_weight2)))

# --- Gamma #2 ---
print("Gamma shape 2:", torch.exp(gamma_shape2))
print("Gamma scale 2:", torch.exp(gamma_scale2) * 500)  # scale factor 500
print("Gamma weight 2:",
      torch.exp(gamma_weight2) / (torch.exp(gamma_weight1) + torch.exp(gamma_weight2)))

# --- von Mises #1 ---
# % (2*np.pi) ensures the mu (angle) is wrapped within [0, 2π)
print("Von Mises mu 1:", vonmises_mu1 % (2*np.pi))
print("Von Mises kappa 1:", torch.exp(vonmises_kappa1))
print("Von Mises weight 1:",
      torch.exp(vonmises_weight1) / (torch.exp(vonmises_weight1) + torch.exp(vonmises_weight2)))

# --- von Mises #2 ---
print("Von Mises mu 2:", vonmises_mu2 % (2*np.pi))
print("Von Mises kappa 2:", torch.exp(vonmises_kappa2))
print("Von Mises weight 2:",
      torch.exp(vonmises_weight2) / (torch.exp(vonmises_weight1) + torch.exp(vonmises_weight2)))


## Plot the movement distributions

We can use the movement parameters to plot the step length and turning angle distributions for the sample covariates.

In [ ]:
# -------------------------------------------------------------------------
# Define helper functions for calculating Gamma and von Mises log-densities
# -------------------------------------------------------------------------
def gamma_density(x, shape, scale):
    """
    Computes the log of the Gamma density for each value in x.

    Args:
      x (Tensor): Input values for which to compute the density.
      shape (float): Gamma shape parameter
      scale (float): Gamma scale parameter

    Returns:
      Tensor: The log of the Gamma probability density at each x.
    """
    return -1*torch.lgamma(shape) - shape*torch.log(scale) \
           + (shape - 1)*torch.log(x) - x/scale

def vonmises_density(x, kappa, vm_mu):
    """
    Computes the log of the von Mises density for each value in x.

    Args:
      x (Tensor): Input angles in radians.
      kappa (float): Concentration parameter (kappa)
      vm_mu (float): Mean direction parameter (mu)

    Returns:
      Tensor: The log of the von Mises probability density at each x.
    """
    return kappa*torch.cos(x - vm_mu) - 1*(np.log(2*torch.pi) + torch.log(torch.special.i0(kappa)))    


# -------------------------------------------------------------------------
# Round and display the mixture weights for the Gamma distributions
# -------------------------------------------------------------------------
gamma_weight1_recovered = torch.exp(gamma_weight1)/(torch.exp(gamma_weight1) + torch.exp(gamma_weight2))
rounded_gamma_weight1 = round(gamma_weight1_recovered.item(), 2)

gamma_weight2_recovered = torch.exp(gamma_weight2)/(torch.exp(gamma_weight1) + torch.exp(gamma_weight2))
rounded_gamma_weight2 = round(gamma_weight2_recovered.item(), 2)

# -------------------------------------------------------------------------
# Round and display the mixture weights for the von Mises distributions
# -------------------------------------------------------------------------
vonmises_weight1_recovered = torch.exp(vonmises_weight1)/(torch.exp(vonmises_weight1) + torch.exp(vonmises_weight2))
rounded_vm_weight1 = round(vonmises_weight1_recovered.item(), 2)

vonmises_weight2_recovered = torch.exp(vonmises_weight2)/(torch.exp(vonmises_weight1) + torch.exp(vonmises_weight2))
rounded_vm_weight2 = round(vonmises_weight2_recovered.item(), 2)


# -------------------------------------------------------------------------
# 1. Plotting the Gamma mixture distribution
#    a) Generate x values
#    b) Compute individual Gamma log densities
#    c) Exponentiate and combine using recovered weights
# -------------------------------------------------------------------------
x_values = torch.linspace(1, 101, 1000).to(device)
gamma1_density = gamma_density(x_values, torch.exp(gamma_shape1), torch.exp(gamma_scale1))
gamma2_density = gamma_density(x_values, torch.exp(gamma_shape2), torch.exp(gamma_scale2)*500)
gamma_mixture_density = gamma_weight1_recovered*torch.exp(gamma1_density) \
                        + gamma_weight2_recovered*torch.exp(gamma2_density)

# Move results to CPU and convert to NumPy for plotting
x_values_np = x_values.cpu().numpy()
gamma1_density_np = np.exp(gamma1_density.cpu().numpy())
gamma2_density_np = np.exp(gamma2_density.cpu().numpy())
gamma_mixture_density_np = gamma_mixture_density.cpu().numpy()

# -------------------------------------------------------------------------
# 2. Plot the Gamma distributions and their mixture
# -------------------------------------------------------------------------
plt.plot(x_values_np, gamma1_density_np, label=f'Gamma 1 Density: weight = {rounded_gamma_weight1}')
plt.plot(x_values_np, gamma2_density_np, label=f'Gamma 2 Density: weight = {rounded_gamma_weight2}')
plt.plot(x_values_np, gamma_mixture_density_np, label='Gamma Mixture Density')
plt.xlabel('x')
plt.ylabel('Density')
plt.title('Gamma Density Function')
plt.legend()
plt.show()


# -------------------------------------------------------------------------
# 3. Plotting the von Mises mixture distribution
#    a) Generate x values from -π to π
#    b) Compute individual von Mises log densities
#    c) Exponentiate and combine using recovered weights
# -------------------------------------------------------------------------
x_values = torch.linspace(-np.pi, np.pi, 1000).to(device)
vonmises1_density = vonmises_density(x_values, torch.exp(vonmises_kappa1), vonmises_mu1)
vonmises2_density = vonmises_density(x_values, torch.exp(vonmises_kappa2), vonmises_mu2)
vonmises_mixture_density = vonmises_weight1_recovered*torch.exp(vonmises1_density) \
                           + vonmises_weight2_recovered*torch.exp(vonmises2_density)

# Move results to CPU and convert to NumPy for plotting
x_values_np = x_values.cpu().numpy()
vonmises1_density_np = np.exp(vonmises1_density.cpu().numpy())
vonmises2_density_np = np.exp(vonmises2_density.cpu().numpy())
vonmises_mixture_density_np = vonmises_mixture_density.cpu().numpy()

# -------------------------------------------------------------------------
# 4. Plot the von Mises distributions and their mixture
# -------------------------------------------------------------------------
plt.plot(x_values_np, vonmises1_density_np, label=f'Von Mises 1 Density: weight = {rounded_vm_weight1}')
plt.plot(x_values_np, vonmises2_density_np, label=f'Von Mises 2 Density: weight = {rounded_vm_weight2}')
plt.plot(x_values_np, vonmises_mixture_density_np, label='Von Mises Mixture Density')
plt.xlabel('x (radians)')
plt.ylabel('Density')
plt.title('Von Mises Density Function')
plt.ylim(0, 0.4)  # Set a limit for the y-axis
plt.legend()
plt.show()


## Generate a distribution of movement parameters

To see how variable the movement parameters are across samples, we can generate a distribution of movement parameters from a batch of samples.

We take the code from above that we used to create the DataLoader for the test data and increase the batch size (to get more samples to create the distribution from).

As we're not using the test dataset any more, we'll just put all of the samples in the same batch, and generate movement parameters for all of them.

In [ ]:
print(f'There are {len(pretraining_dataset_test)} samples in the test dataset')
bs = len(pretraining_dataset_test) # batch size
bs = 100
dataloader_test = DataLoader(dataset=pretraining_dataset_test, batch_size=bs, shuffle=True)

Take all of the samples from the test dataset and put them in a single batch.

In [ ]:
# -----------------------------------------------------------
# Fetch a batch of data from the training dataloader
# -----------------------------------------------------------
x1_batch, x2_batch, x3_batch, labels, _ = next(iter(dataloader_test))

x1_batch = x1_batch.to(device)
x2_batch = x2_batch.to(device)
x3_batch = x3_batch.to(device)
# labels = labels.to(device)

# -----------------------------------------------------------
# Register a forward hook to capture the outputs
#    from 'fcn_movement_all' during the forward pass
# -----------------------------------------------------------
hook_handle = model_pretraining.fcn_movement_all.register_forward_hook(hook)

# -----------------------------------------------------------
# Perform a forward pass in evaluation mode to generate
#    and capture the sub-network's outputs in 'intermediate_output'
# -----------------------------------------------------------
model_pretraining.eval()  # Disables certain layers like dropout

# Pass the batch through the model
final_output = model_pretraining((x1_batch, x2_batch, x3_batch))

# -----------------------------------------------------------
# Prepare lists to store the distribution parameters
#    for each sample in the batch
# -----------------------------------------------------------
gamma_shape1_list = []
gamma_scale1_list = []
gamma_weight1_list = []
gamma_shape2_list = []
gamma_scale2_list = []
gamma_weight2_list = []
vonmises_mu1_list = []
vonmises_kappa1_list = []
vonmises_weight1_list = []
vonmises_mu2_list = []
vonmises_kappa2_list = []
vonmises_weight2_list = []

# -----------------------------------------------------------
# Extract parameters from 'intermediate_output'
#    for every sample in the batch
# -----------------------------------------------------------
for batch_output in intermediate_output:
    # Each 'batch_output' corresponds to one forward pass;
    # it might contain multiple samples if the batch size > 1
    for sample_output in batch_output:
        # Unpack the 12 parameters of the Gamma and von Mises mixtures
        gamma_shape1, gamma_scale1, gamma_weight1, \
        gamma_shape2, gamma_scale2, gamma_weight2, \
        vonmises_mu1, vonmises_kappa1, vonmises_weight1, \
        vonmises_mu2, vonmises_kappa2, vonmises_weight2 = sample_output

        # Convert log-space parameters to real space, then store
        gamma_shape1_list.append(torch.exp(gamma_shape1).item())
        gamma_scale1_list.append(torch.exp(gamma_scale1).item())
        gamma_weight1_list.append(
            (torch.exp(gamma_weight1)/(torch.exp(gamma_weight1) + torch.exp(gamma_weight2))).item()
        )
        gamma_shape2_list.append(torch.exp(gamma_shape2).item())
        gamma_scale2_list.append((torch.exp(gamma_scale2)*500).item())  # scale factor 500
        gamma_weight2_list.append(
            (torch.exp(gamma_weight2)/(torch.exp(gamma_weight1) + torch.exp(gamma_weight2))).item()
        )
        vonmises_mu1_list.append((vonmises_mu1 % (2*np.pi)).item())
        vonmises_kappa1_list.append(torch.exp(vonmises_kappa1).item())
        vonmises_weight1_list.append(
            (torch.exp(vonmises_weight1)/(torch.exp(vonmises_weight1) + torch.exp(vonmises_weight2))).item()
        )
        vonmises_mu2_list.append((vonmises_mu2 % (2*np.pi)).item())
        vonmises_kappa2_list.append(torch.exp(vonmises_kappa2).item())
        vonmises_weight2_list.append(
            (torch.exp(vonmises_weight2)/(torch.exp(vonmises_weight1) + torch.exp(vonmises_weight2))).item()
        )


### Plot the distribution of movement parameters

In [ ]:
# -----------------------------------------------------------
# Define a helper function to plot histograms
#    for the collected parameters
# -----------------------------------------------------------
def plot_histogram(data, title, xlabel):
    """
    Plots a histogram of the provided data.

    Args:
        data (list): Data points to plot in a histogram.
        title (str): Title of the histogram plot.
        xlabel (str): X-axis label.
    """
    plt.figure()
    plt.hist(data, bins=30, alpha=0.75)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel('Frequency')
    plt.show()

# -----------------------------------------------------------
# Plot histograms for each parameter distribution
# -----------------------------------------------------------
plot_histogram(gamma_shape1_list, 'Gamma Shape 1 Distribution', 'Shape 1')
plot_histogram(gamma_scale1_list, 'Gamma Scale 1 Distribution', 'Scale 1')
plot_histogram(gamma_weight1_list, 'Gamma Weight 1 Distribution', 'Weight 1')
plot_histogram(gamma_shape2_list, 'Gamma Shape 2 Distribution', 'Shape 2')
plot_histogram(gamma_scale2_list, 'Gamma Scale 2 Distribution', 'Scale 2')
plot_histogram(gamma_weight2_list, 'Gamma Weight 2 Distribution', 'Weight 2')
plot_histogram(vonmises_mu1_list, 'Von Mises Mu 1 Distribution', 'Mu 1')
plot_histogram(vonmises_kappa1_list, 'Von Mises Kappa 1 Distribution', 'Kappa 1')
plot_histogram(vonmises_weight1_list, 'Von Mises Weight 1 Distribution', 'Weight 1')
plot_histogram(vonmises_mu2_list, 'Von Mises Mu 2 Distribution', 'Mu 2')
plot_histogram(vonmises_kappa2_list, 'Von Mises Kappa 2 Distribution', 'Kappa 2')
plot_histogram(vonmises_weight2_list, 'Von Mises Weight 2 Distribution', 'Weight 2')

# -----------------------------------------------------------
# Remove the hook to stop capturing outputs
#    in subsequent forward passes
# -----------------------------------------------------------
hook_handle.remove()

# Next-step probability values

We can now calculate the next-step probabilities for each observed step. As we generate habitat selection, movement and next-step probability surfaces, we can get the predicted probability values for each one, which can be compared to the respective process in the SSF.

The process for generating the next-step probabilities is as follows:

1. Get the current location of the individual
2. Crop out the local layers for the current location
3. Run the model of the local layers to get the habitat selection, movement and next-step probability surfaces
4. Get the predicted probability values at the location of the next step
5. Store the predicted probability values and export them as a csv for comparison with the SSF

First, select the data to generate prediction values for. For testing the function we can select a subset.

### Create directory to save prediction outputs

In [ ]:
prediction_training_data_dir = f'{output_dir}/prediction_training_data_images'
os.makedirs(prediction_training_data_dir, exist_ok=True)

In [ ]:
prediction_batch_size = 32

# To select the full dataset for predictions
prediction_dataset = pretraining_dataset

# To select a subset of the dataset for predictions
# prediction_dataset = torch.utils.data.Subset(pretraining_dataset, list(range(0, 10000)))

n_samples = len(prediction_dataset)
print(f'Number of samples: {n_samples}')

prediction_dataloader = DataLoader(
    dataset=prediction_dataset,
    batch_size=prediction_batch_size,  # Batching still helps for CPU-GPU transfer efficiency
    shuffle=False
)

print(f'Number of batches: {len(prediction_dataloader)}') # number of batches

loss_fn_pred = deepSSF_loss.negativeLogLikeLoss(reduction='none')

### Create empty vectors to store the predicted probabilities
habitat_log_probs = np.zeros(n_samples)
move_log_probs = np.zeros(n_samples)
next_step_log_probs = np.zeros(n_samples)

## Loop over each step

In [ ]:
# print(f'Model device: {model.device}')

model_pretraining.eval()  # Set once before the loop
with torch.no_grad():

    for batch_idx, (x1, x2, x3, y, raster_transform) in enumerate(prediction_dataloader):

        # Move the batch of data to the specified device (CPU/GPU)
        x1 = x1.to(device)
        x2 = x2.to(device)
        x3 = x3.to(device)

        # Forward pass: compute the model output and loss
        outputs = model_pretraining((x1, x2, x3))
        total_loss, habitat_loss, movement_loss = loss_fn_pred(outputs, y)

        # print(total_loss.detach().cpu().numpy()) # when reduction='none' all individual losses are returned

        # Store the individual log-probabilities for each sample in the batch
        ## Create indices to place batch results in the vectors of log-probabilities
        start_idx = batch_idx * batch_size
        end_idx = start_idx + x1.size(0)  # Actual batch size (may be smaller for the last batch)
        ## Store the negative losses (log-probabilities)
        habitat_log_probs[start_idx:end_idx] = -habitat_loss.detach().cpu().numpy()
        move_log_probs[start_idx:end_idx] = -movement_loss.detach().cpu().numpy()
        next_step_log_probs[start_idx:end_idx] = -total_loss.detach().cpu().numpy()

        if batch_idx == 0:  # Only plot the first batch

            # Plotting
            sample_spatial_covs = x1.detach().cpu().numpy()
            sample_temporal_covs = x2
            sample_prev_bearing = x3
            sample_next_step =  y

            # print(f'Shape of the sample spatial covariates:  {sample_spatial_covs.shape}')
            # print(f'Shape of the sample temporal covariates: {sample_temporal_covs.shape}')
            # print(f'Shape of the sample previous bearing:    {sample_prev_bearing.shape}')
            # # Print the pixel coordinates for the next step
            # print(f'Location of sample next step (px, py):   {sample_next_step}')     

            for i in range(x1.size(0)):  # Loop over samples in the batch

                print(f'Processing sample {i+1} in the batch of size {x1.size(0)}')
                print(f'Index in the full dataset: {start_idx + i}')

                # Get the current sample's next step coordinates
                sample_next_step_coords = sample_next_step

                # Pull out the RGB layers for plotting
                blue_layer = sample_spatial_covs[i,1,:,:]
                green_layer = sample_spatial_covs[i,2,:,:]
                red_layer = sample_spatial_covs[i,3,:,:]
                # Stack the RGB layers
                rgb_image_np = np.stack([red_layer, green_layer, blue_layer], axis=-1)
                # Normalize to the range [0, 1] for display
                rgb_image_np = (rgb_image_np - rgb_image_np.min()) / (rgb_image_np.max() - rgb_image_np.min())

                # Extract and exponentiate the habitat density channel
                hab_density = outputs.detach().cpu().numpy()[i, :, :, 0]
                move_density = outputs.detach().cpu().numpy()[i, :, :, 1]
                next_step_density = hab_density + move_density

                # Set the location of the next step to NaN in the covariate layers
                sample_spatial_covs[i, :, sample_next_step[1][i], sample_next_step[0][i]] = -np.inf
                rgb_image_np[sample_next_step[1][i], sample_next_step[0][i], :] = 1.0 # set to 1 for white in RGB image
                ## Prediction surfaces
                hab_density[sample_next_step[1][i], sample_next_step[0][i]] = -np.inf
                move_density[sample_next_step[1][i], sample_next_step[0][i]] = -np.inf
                next_step_density[sample_next_step[1][i], sample_next_step[0][i]] = -np.inf

                # Plot the subset
                fig, axs = plt.subplots(2, 2, figsize=(10, 10))

                # Plot RGB
                im1 = axs[0, 0].imshow(rgb_image_np)
                axs[0, 0].set_title('Sentinel-2 RGB')

                # Plot slope
                im2 = axs[0, 1].imshow(sample_spatial_covs[0,13,:,:], cmap='viridis')
                axs[0, 1].set_title('Slope')
                # fig.colorbar(im2, ax=axs[0, 1], shrink=0.7)

                # Plot habitat selection
                im3 = axs[1, 0].imshow(hab_density * x_mask * y_mask, cmap='viridis')
                axs[1, 0].set_title('Habitat selection log-probability')
                # fig.colorbar(im3, ax=axs[1, 0], shrink=0.7)

                # # Movement density (change the axis and uncomment one of the other panels)
                # im3 = axs[1, 0].imshow(move_density_mask, cmap='viridis')
                # axs[1, 0].set_title('Movement log-probability')
                # fig.colorbar(im3, ax=axs[0, 1], shrink=0.7)

                # Next-step probability
                im4 = axs[1, 1].imshow(next_step_density * x_mask * y_mask, cmap='viridis')
                axs[1, 1].set_title('Next-step log-probability')
                # fig.colorbar(im4, ax=axs[1, 1], shrink=0.7)

                if i < 1:
                    # Save the figure
                    filename_covs = f'{prediction_training_data_dir}/prediction_fitted_data_index_{start_idx + i}.png'
                    plt.tight_layout()
                    plt.savefig(filename_covs, dpi=150, bbox_inches='tight')
                    plt.show()
                    plt.close()  # Close the figure to free memory
                else:
                    # Save the figure
                    filename_covs = f'{prediction_training_data_dir}/prediction_fitted_data_index_{start_idx + i}.png'
                    plt.tight_layout()
                    plt.savefig(filename_covs, dpi=150, bbox_inches='tight')
                    # plt.show()
                    plt.close()  # Close the figure to free memory

# Create a DataFrame to hold the log-probabilities
df_probs_pretraining = pd.DataFrame({
    'habitat_log_prob': habitat_log_probs,
    'movement_log_prob': move_log_probs,
    'next_step_log_prob': next_step_log_probs,
    'habitat_prob': np.exp(habitat_log_probs),
    'movement_prob': np.exp(move_log_probs),
    'next_step_prob': np.exp(next_step_log_probs)
})

df_probs_pretraining
# df_probs_pretraining.head()

# Save the log-probabilities to a CSV file
df_probs_pretraining.to_csv(f'{output_dir}/test_probs_pretraining.csv', index=False)

## Calculate the null probabilities

As each cell has a probability values, we can calculate what the probability would be if the model provided no information at all, and each cell was equally likely to be the next step. This is just 1 divided by the total number of cells.

In [ ]:
null_prob = 1 / (window_size ** 2)
print(f'Null probability: {null_prob:.3e}')

## Compute the rolling average of the probabilities

In [ ]:
rolling_window_size = 100 # Rolling window size

# Convert to pandas Series and compute rolling mean
rolling_mean_habitat = df_probs_pretraining['habitat_prob'].rolling(window=rolling_window_size, center=True).mean()
rolling_mean_movement = df_probs_pretraining['movement_prob'].rolling(window=rolling_window_size, center=True).mean()
rolling_mean_next_step = df_probs_pretraining['next_step_prob'].rolling(window=rolling_window_size, center=True).mean()

# print(rolling_mean_habitat)

## Average of all probabilities

In [ ]:
average_habitat_prob = df_probs_pretraining['habitat_prob'].mean()
average_move_prob = df_probs_pretraining['movement_prob'].mean()
average_next_step_prob = df_probs_pretraining['next_step_prob'].mean()

print(f'Average habitat probability:    {average_habitat_prob:.3e}')
print(f'Comparison to null:          {average_habitat_prob / null_prob:.3f}x \n')
print(f'Average movement probability:   {average_move_prob:.3e}')
print(f'Comparison to null:          {average_move_prob / null_prob:.3f}x \n')
print(f'Average next-step probability:  {average_next_step_prob:.3e}')
print(f'Comparison to null:          {average_next_step_prob / null_prob:.3f}x \n')

# Plot the probabilities

We can get an idea of how variable the probabilities are for the habitat selection and movement surfaces, and for the next-step probabilities, by plotting them across the trajectory

In [ ]:
# Plot the habitat probs through time as a line graph
plt.plot(df_probs_pretraining['habitat_prob'][:100], color='blue', label='Habitat Probabilities - S2')
plt.plot(rolling_mean_habitat[:100], color='red', label='Rolling Mean')
plt.axhline(y=null_prob, color='black', linestyle='--', label='Null Probability')  # null probs
plt.xlabel('Index')
plt.ylabel('Probability')
plt.title('Habitat Probability')
plt.legend()  # Add legend to differentiate lines
plt.savefig(f'{output_dir}/habitat_probs_100_steps.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot the habitat probs through time as a line graph
plt.plot(df_probs_pretraining['habitat_prob'], color='blue', label='Habitat Probabilities - S2')
plt.plot(rolling_mean_habitat[rolling_mean_habitat > 0], color='red', label='Rolling Mean')
plt.axhline(y=null_prob, color='black', linestyle='--', label='Null Probability')  # null probs
plt.xlabel('Index')
plt.ylabel('Probability')
# plt.ylim(0, 5e-4)  # Set a limit for the y-axis
plt.title('Habitat Probability')
plt.legend()  # Add legend to differentiate lines
plt.savefig(f'{output_dir}/habitat_probs.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot the movement probs through time as a line graph
plt.plot(df_probs_pretraining['movement_prob'], color='blue', label='Movement Probabilities - S2')
plt.plot(rolling_mean_movement[rolling_mean_movement > 0], color='red', label='Rolling Mean')
plt.axhline(y=null_prob, color='black', linestyle='--', label='Null Probability')  # null probs
plt.xlabel('Index')
plt.ylabel('Probability')
plt.title('Movement Probability')
plt.legend()  # Add legend to differentiate lines
plt.savefig(f'{output_dir}/move_probs.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot the next step probs through time as a line graph
plt.plot(df_probs_pretraining['next_step_prob'], color='blue', label='Next Step Probabilities - S2')
plt.plot(rolling_mean_next_step[rolling_mean_next_step > 0], color='red', label='Rolling Mean')
plt.axhline(y=null_prob, color='black', linestyle='--', label='Null Probability')  # null probs
plt.xlabel('Index')
plt.ylabel('Probability')
plt.title('Next Step Probability')
plt.legend()  # Add legend to differentiate lines
plt.savefig(f'{output_dir}/next_step_probs.png', dpi=300, bbox_inches='tight')
plt.show()

### Make a GIF of the prediction images

In [ ]:
# Path to your images
image_folder =  f'{output_dir}/prediction_fitted_data_images'
# Output GIF filename
output_filename_gif = f'{output_dir}/prediction_fitted_data_steps.gif'
# Create the GIF
create_gif(image_folder, output_filename_gif, fps=3)

# Output MP4 filename
output_filename_mp4 = f'{output_dir}/prediction_fitted_data_steps.mp4'
# Create the MP4
create_gif(image_folder, output_filename_mp4, fps=3)